In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Hemispheric Dissociation of Multiscale Autonomic Complexity and Cardiovagal Reserve
Predicts Orthostatic Cerebral Perfusion and Gait Speed in Chronic Ischemic Stroke

Colab & Kaggle Ready | Crash-Proof | CCS HRV Pipeline Compliant (method='nabian2018')
with Upfront 1000 Hz LabVIEW Clock Calibration for PhysioNet CVES (v1.0.0).

Author: Aswin Tony Joseph
License: MIT
"""

import importlib
import shutil
import subprocess
import sys

# =============================================================================
# 0A. AUTO-INSTALL MISSING DEPENDENCIES (COLAB / KAGGLE SAFE)
# =============================================================================
REQUIRED_PACKAGES = {
    "awscli": "awscli",
    "wfdb": "wfdb",
    "neurokit2": "neurokit2",
    "statsmodels": "statsmodels",
    "seaborn": "seaborn",
    "tqdm": "tqdm",
}


def ensure_dependencies() -> None:
    missing = []
    for mod_name, pip_name in REQUIRED_PACKAGES.items():
        if mod_name == "awscli":
            if shutil.which("aws") is None:
                missing.append(pip_name)
        else:
            try:
                importlib.import_module(mod_name)
            except ImportError:
                missing.append(pip_name)
    if missing:
        print(f"[SETUP] Installing missing packages: {', '.join(missing)} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
        print("[SETUP] Dependencies installed successfully.")


ensure_dependencies()

import gc
import logging
import os
import traceback
import warnings
import zipfile
from pathlib import Path
import patsy 
from statsmodels.stats.outliers_influence import variance_inflation_factor

import matplotlib
matplotlib.rcParams["agg.path.chunksize"] = 10000
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import neurokit2 as nk
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import wfdb
from scipy import stats
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# =============================================================================
# 0B. ENVIRONMENT AUTO-DETECTION (KAGGLE vs. GOOGLE COLAB vs. LOCAL)
# =============================================================================
IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = Path("/content").exists() and not IS_KAGGLE
ROOT_WORK = Path("/kaggle/working") if IS_KAGGLE else (Path("/content") if IS_COLAB else Path.cwd())

CONFIG = {
    "BASE_DIR": ROOT_WORK / "cves_nabian2018_ccs_workspace",
    "DATA_DIR": ROOT_WORK / "cves-dataset" / "data",
    "SUBJECTS_CSV": ROOT_WORK / "cves-dataset" / "subjects.csv",
    "DOWNLOAD_DATA": False,           # Set False if PhysioNet S3 data already synced
    "RUN_EXTRACTION": False,          # Set False to skip WFDB extraction and load existing CSV
    "SAVE_QC_PLOTS": False,          # Set True on Kaggle (or Colab + Drive) to render ~750 QC PNGs
    "QC_DPI": 300,                   # 150 DPI prevents RAM/disk bloat; use 300 on Kaggle if desired
    "SQI_THRESHOLD": 0.50,           # Hard SQI threshold (CCS standard)
    "RR_MIN_MS": 300.0,              # 200 BPM upper physiological bound
    "RR_MAX_MS": 1500.0,             # 40 BPM lower physiological bound
    "BAD_RR_THRESHOLD_PCT": 10.0,    # >10% implausible RR intervals flagged 'POOR'
    "ZIP_FILENAME": "CVES_Publisher_Tables_Figures_Nabian2018.zip",
    "TRIGGER_COLAB_DOWNLOAD": True,  # Auto-downloads lightweight ZIP in Colab
}

OUT_DIR = CONFIG["BASE_DIR"]
FIG_DIR = OUT_DIR / "Figures_Publication"
SUPP_FIG_DIR = OUT_DIR / "Figures_Supplementary"
TABLE_DIR = OUT_DIR / "Tables_CSV"
TACHO_DIR = OUT_DIR / "Supplementary_Tachograms"
QC_DIR = OUT_DIR / "Supplementary_QC_Plots"

for d in [OUT_DIR, FIG_DIR, SUPP_FIG_DIR, TABLE_DIR, TACHO_DIR, QC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(OUT_DIR / "run_log.txt", mode="w"),
    ],
)
log = logging.getLogger("CVES_CCS_Nabian2018")
log.info("Environment: %s | Workspace: %s", "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "Local"), OUT_DIR)


# =============================================================================
# 1. PHYSIONET CVES DATA ACQUISITION
# =============================================================================
def download_cves_dataset() -> None:
    CONFIG["DATA_DIR"].mkdir(parents=True, exist_ok=True)
    CONFIG["SUBJECTS_CSV"].parent.mkdir(parents=True, exist_ok=True)

    log.info("Downloading subjects.csv from PhysioNet S3 (cves v1.0.0)...")
    subprocess.run(
        [
            "aws", "s3", "cp", "--no-sign-request",
            "s3://physionet-open/cves/1.0.0/subjects.csv",
            str(CONFIG["SUBJECTS_CSV"]),
        ],
        check=True,
    )

    log.info("Syncing sit-stand, head-up-tilt, and sit-stand-balance WFDB files...")
    subprocess.run(
        [
            "aws", "s3", "sync", "--no-sign-request",
            "s3://physionet-open/cves/1.0.0/data",
            str(CONFIG["DATA_DIR"]),
            "--exclude", "*",
            "--include", "sit-stand/*",
            "--include", "head-up-tilt/*",
            "--include", "sit-stand-balance/*",
        ],
        check=True,
    )
    log.info("PhysioNet cves dataset sync complete.")


# =============================================================================
# 2. HARDWARE-LOCKED EPOCH SEGMENTATION & CCS HRV PROCESSING (NABIAN2018)
# =============================================================================
def compute_prsa_dc_ac(rr_ms: np.ndarray) -> tuple:
    rr = np.asarray(rr_ms, dtype=float)
    if len(rr) < 20:
        return np.nan, np.nan
    idx = np.arange(2, len(rr) - 1)
    denom = np.where(rr[idx - 1] == 0, np.nan, rr[idx - 1])
    valid = (np.abs(rr[idx] - rr[idx - 1]) / denom) < 0.20

    dec = idx[(rr[idx] > rr[idx - 1]) & valid]
    acc = idx[(rr[idx] < rr[idx - 1]) & valid]

    dc = float(np.mean((rr[dec] + rr[dec + 1] - rr[dec - 1] - rr[dec - 2]) / 4.0)) if len(dec) > 0 else np.nan
    ac = float(np.mean((rr[acc] + rr[acc + 1] - rr[acc - 1] - rr[acc - 2]) / 4.0)) if len(acc) > 0 else np.nan
    return dc, ac


def compute_coarse_mse_6_15(rr_ms: np.ndarray, m: int = 2, r_factor: float = 0.20) -> float:
    rr = np.asarray(rr_ms, dtype=float)
    rr = rr[np.isfinite(rr)]
    if len(rr) < 100 or np.std(rr) == 0:
        return np.nan
    tol = r_factor * float(np.std(rr, ddof=1))
    vals = []
    for tau in range(6, 16):
        n_blocks = len(rr) // tau
        if n_blocks < 15:
            continue
        cg = rr[: n_blocks * tau].reshape(n_blocks, tau).mean(axis=1)
        try:
            se, _ = nk.entropy_sample(cg, delay=1, dimension=m, tolerance=tol)
            if np.isfinite(se):
                vals.append(float(se))
        except Exception:
            continue
    return float(np.mean(vals)) if vals else np.nan


def get_marker_blocks(rec_path: str, header, true_fs: float) -> list:
    chs = [c.lower() for c in header.sig_name]
    bounds = [0.0]
    if "marker" in chs:
        try:
            m_sig = wfdb.rdrecord(rec_path, channels=[chs.index("marker")]).p_signal[:, 0]
            thresh = np.nanstd(m_sig) * 0.4
            if thresh > 1e-6:
                for j in np.where(np.abs(np.diff(m_sig)) > thresh)[0] / true_fs:
                    if j - bounds[-1] > 10.0:
                        bounds.append(float(j))
        except Exception:
            pass
    total_dur = header.sig_len / true_fs
    if total_dur - bounds[-1] > 10.0:
        bounds.append(float(total_dur))
    return [
        (bounds[i], bounds[i + 1], bounds[i + 1] - bounds[i])
        for i in range(len(bounds) - 1)
        if (bounds[i + 1] - bounds[i]) >= 30.0
    ]


def extract_verified_epochs(rec_path: str, folder_name: str, header, true_fs: float) -> list:
    f_low = folder_name.lower()
    chs = [c.lower() for c in header.sig_name]
    blocks = get_marker_blocks(rec_path, header, true_fs)
    epochs = []

    if f_low == "sit-stand":
        long_blocks = [b for b in blocks if b[2] >= 180.0]
        if len(long_blocks) >= 2:
            b0, b1 = long_blocks[0], long_blocks[1]
            epochs.append(("State0_SeatedRest", 0, b0[0] + 10.0, min(b0[1] - 5.0, b0[0] + 310.0)))
            epochs.append(("State1_SitToStand", 1, b1[0] + 10.0, min(b1[1] - 5.0, b1[0] + 310.0)))

    elif f_low == "head-up-tilt":
        supine = [b for b in blocks if b[0] < 100.0 and b[2] >= 300.0]
        if supine:
            s0 = supine[0][0] + 120.0 if supine[0][2] >= 450.0 else supine[0][0] + 20.0
            epochs.append(("State0_SupineRest", 0, s0, min(supine[0][1] - 20.0, s0 + 300.0)))
        tilt = [b for b in blocks if b[0] >= 1200.0 and b[2] >= 360.0]
        if not tilt:
            tilt = [b for b in blocks if b[0] >= 1000.0 and b[2] >= 240.0]
        if tilt:
            s1 = tilt[-1][0] + 60.0 if tilt[-1][2] >= 400.0 else tilt[-1][0] + 15.0
            epochs.append(("State1_HeadUpTilt", 1, s1, min(tilt[-1][1] - 15.0, s1 + 300.0)))

    elif f_low == "sit-stand-balance":
        stand = []
        if "fz" in chs:
            try:
                fz_sig = wfdb.rdrecord(rec_path, channels=[chs.index("fz")]).p_signal[:, 0]
                for t0, t1, dur in blocks:
                    mean_fz = float(np.nanmean(np.abs(fz_sig[int(t0 * true_fs):int(t1 * true_fs)])))
                    if dur >= 120.0 and mean_fz > 50.0:
                        stand.append((t0, t1, dur))
            except Exception:
                pass
        if len(stand) >= 2:
            epochs.append(("State2_StandEO_Balance", 2, stand[0][0] + 10.0, min(stand[0][1] - 5.0, stand[0][0] + 310.0)))
            epochs.append(("State3_StandEC_DualBalance", 3, stand[1][0] + 10.0, min(stand[1][1] - 5.0, stand[1][0] + 310.0)))

    return epochs


def process_epoch_ccs_nabian(
    ecg_raw_signal: np.ndarray,
    srate: float,
    fname: str,
    sub_id: str,
    session_id: str,
    task_name: str,
    stress_score: int,
):
    # 1. Polarity correction
    ecg_signal, is_inverted = nk.ecg_invert(ecg_raw_signal, sampling_rate=srate, show=False)

    # 2. Primary ECG processing
    signals, ecg_info = nk.ecg_process(ecg_signal, sampling_rate=srate)
    if len(ecg_info["ECG_R_Peaks"]) < 30:
        return None

    # 3. Signal Quality Index (SQI) gating
    quality_scores = nk.ecg_quality(
        signals["ECG_Clean"],
        rpeaks=ecg_info["ECG_R_Peaks"],
        sampling_rate=srate,
    )
    mean_sqi = float(np.mean(quality_scores))
    if mean_sqi < CONFIG["SQI_THRESHOLD"]:
        log.warning("SQI %.3f < %.2f — skipping %s.", mean_sqi, CONFIG["SQI_THRESHOLD"], fname)
        return None

    # 4. Lipponen-Tarvainen artifact correction + Nabian2018 R-peak detector
    _, peaks_info = nk.ecg_peaks(
        signals["ECG_Clean"],
        correct_artifacts=True,
        method="nabian2018",
        sampling_rate=srate,
    )

    # 5. SAFE integer casting & bounds clipping (prevents KeyError / IndexError)
    raw_peaks = np.asarray(peaks_info["ECG_R_Peaks"], dtype=float)
    raw_peaks = raw_peaks[np.isfinite(raw_peaks)]
    valid_peaks = np.unique(raw_peaks.astype(int))
    valid_peaks = valid_peaks[(valid_peaks >= 0) & (valid_peaks < len(signals))]
    if len(valid_peaks) < 30:
        return None

    ecg_info["ECG_R_Peaks"] = valid_peaks
    peaks_info["ECG_R_Peaks"] = valid_peaks
    signals["ECG_R_Peaks"] = 0
    signals.iloc[valid_peaks, signals.columns.get_loc("ECG_R_Peaks")] = 1

    # 6. RR intervals & physiological bounds check (300-1500 ms)
    rr_ms = np.diff(valid_peaks) / srate * 1000.0
    pd.DataFrame(rr_ms, columns=["RR_Intervals_ms"]).to_csv(TACHO_DIR / f"{fname}.csv", index=False)

    bad_mask = (rr_ms < CONFIG["RR_MIN_MS"]) | (rr_ms > CONFIG["RR_MAX_MS"])
    n_rr_total = int(len(rr_ms))
    n_rr_bad = int(np.sum(bad_mask))
    pct_bad = float(100.0 * n_rr_bad / n_rr_total) if n_rr_total > 0 else 100.0
    data_quality_flag = "POOR" if pct_bad > CONFIG["BAD_RR_THRESHOLD_PCT"] else "OK"

    # 7. Isolated ECG-Derived Respiration (EDR) — never crashes HRV extraction
    mean_rsp_rate, sd_rsp_rate = np.nan, np.nan
    try:
        ecg_rate_signal = nk.ecg_rate(peaks_info, sampling_rate=srate, desired_length=len(ecg_signal))
        edr_signal = nk.ecg_rsp(ecg_rate_signal, sampling_rate=srate)
        rsp_signals, _ = nk.rsp_process(edr_signal, sampling_rate=srate)
        mean_rsp_rate = float(np.mean(rsp_signals["RSP_Rate"]))
        sd_rsp_rate = float(np.std(rsp_signals["RSP_Rate"], ddof=1))
    except Exception as exc:
        log.warning("EDR extraction failed for %s (continuing with NaN): %s", fname, exc)

    # 8. Compute HRV + Optional CCS QC figures
    if CONFIG["SAVE_QC_PLOTS"]:
        epoch_qc_dir = QC_DIR / fname
        epoch_qc_dir.mkdir(parents=True, exist_ok=True)
        hrv_df = nk.hrv(ecg_info, sampling_rate=srate, show=True)
        hrv_fig = plt.gcf()
        hrv_fig.set_size_inches(16.0, 9.0)
        hrv_fig.savefig(epoch_qc_dir / f"{fname}_hrv_summary.png", dpi=CONFIG["QC_DPI"])
        plt.close("all")

        nk.ecg_plot(signals, info=ecg_info)
        qual_fig = plt.gcf()
        qual_fig.set_size_inches(16.0, 9.0)
        qual_fig.savefig(epoch_qc_dir / f"{fname}_quality_check.png", dpi=CONFIG["QC_DPI"])
        plt.close("all")
    else:
        hrv_df = nk.hrv(ecg_info, sampling_rate=srate, show=False)

    dc_val, ac_val = compute_prsa_dc_ac(rr_ms)
    mse_coarse = compute_coarse_mse_6_15(rr_ms)

    hrv_df["filename"] = fname
    hrv_df["Subject"] = sub_id
    hrv_df["Session"] = session_id
    hrv_df["Task"] = task_name
    hrv_df["Stress_Score"] = stress_score
    hrv_df["true_srate_hz"] = srate
    hrv_df["data_quality_flag"] = data_quality_flag
    hrv_df["mean_sqi"] = round(mean_sqi, 4)
    hrv_df["n_rpeaks"] = len(valid_peaks)
    hrv_df["n_rr_total"] = n_rr_total
    hrv_df["n_rr_bad"] = n_rr_bad
    hrv_df["pct_rr_bad"] = round(pct_bad, 2)
    hrv_df["is_inverted"] = is_inverted
    hrv_df["mean_resp_rate"] = mean_rsp_rate
    hrv_df["sd_resp_rate"] = sd_rsp_rate
    hrv_df["PRSA_DC_ms"] = round(dc_val, 4) if not np.isnan(dc_val) else np.nan
    hrv_df["PRSA_AC_ms"] = round(ac_val, 4) if not np.isnan(ac_val) else np.nan
    hrv_df["HRV_MSE_Coarse_6_15"] = round(mse_coarse, 4) if not np.isnan(mse_coarse) else np.nan
    return hrv_df


def run_extraction_pipeline() -> pd.DataFrame:
    def run_epoch_sanity_check(master_df: pd.DataFrame) -> pd.DataFrame:
    """Audits extracted epoch HR & physiological bounds against subjects.csv ground truth."""
    pheno = pd.read_csv(CONFIG["SUBJECTS_CSV"])
    pheno.columns = [c.strip() for c in pheno.columns]
    subj_col = next((c for c in pheno.columns if "sub" in c.lower() or "id" in c.lower()), pheno.columns[0])

    df = master_df.copy()
    df["Subject"] = df["Subject"].astype(str).str.lower().str.strip()
    pheno["Subject"] = pheno[subj_col].astype(str).str.lower().str.strip()
    m = df.merge(pheno, on="Subject", how="inner")

    gt_pairs = [
        ("State0_SupineRest", "(Baseline Mean) HR BP BASELINE"),
        ("State1_HeadUpTilt", "(Tilt mn) HR TILT"),
        ("State2_StandEO_Balance", "(StandEO mn) Mean HR StandEO"),
        ("State3_StandEC_DualBalance", "(StandEC mn) Mean HR StandEC"),
    ]

    summary_rows, outlier_rows = [], []
    for task, gt_col in gt_pairs:
        if gt_col not in m.columns:
            continue
        sub = m[m["Task"] == task][["Subject", "HR_bpm", gt_col, "HRV_RMSSD", "HRV_DFA_alpha1", "mean_sqi", "pct_rr_bad"]].copy()
        sub[gt_col] = pd.to_numeric(sub[gt_col], errors="coerce")
        sub = sub.dropna(subset=["HR_bpm", gt_col])
        if len(sub) < 5:
            continue

        diff = sub["HR_bpm"] - sub[gt_col]
        r_p, _ = stats.pearsonr(sub["HR_bpm"], sub[gt_col])
        r_s, _ = stats.spearmanr(sub["HR_bpm"], sub[gt_col])
        summary_rows.append({
            "Task": task, "GT_Column": gt_col, "N": len(sub),
            "Pearson_r": round(r_p, 4), "Spearman_rs": round(r_s, 4),
            "MAE_bpm": round(float(np.mean(np.abs(diff))), 2),
            "Bias_bpm": round(float(np.mean(diff)), 2),
            "Med_RMSSD_ms": round(float(sub["HRV_RMSSD"].median()), 2),
            "Med_DFA_a1": round(float(sub["HRV_DFA_alpha1"].median()), 3),
            "N_Diff_gt5bpm": int((np.abs(diff) > 5.0).sum()),
            "Pass_Check": "PASS" if (r_s >= 0.85 and sub["HRV_RMSSD"].median() < 50.0) else "REVIEW",
        })
        for _, row in sub[np.abs(diff) > 5.0].iterrows():
            outlier_rows.append({
                "Task": task, "Subject": row["Subject"],
                "Extracted_HR": round(row["HR_bpm"], 2), "GT_HR": round(row[gt_col], 2),
                "Diff_bpm": round(row["HR_bpm"] - row[gt_col], 2),
                "RMSSD_ms": round(row["HRV_RMSSD"], 2), "SQI": round(row["mean_sqi"], 3),
            })

    audit_df = pd.DataFrame(summary_rows)
    audit_df.to_csv(TABLE_DIR / "Table_S0_Epoch_Sanity_Audit.csv", index=False)
    if outlier_rows:
        pd.DataFrame(outlier_rows).to_csv(TABLE_DIR / "Table_S0B_HR_Outliers_gt5bpm.csv", index=False)

    log.info("=== EPOCHING & PROCESSING SANITY CHECK ===\n%s", audit_df.to_string(index=False))
    return audit_df
    out_csv = TABLE_DIR / "Table_S1_Master_Epoch_Features_Nabian2018_1000Hz.csv"
    hea_files = [
        f for f in sorted(CONFIG["DATA_DIR"].glob("**/*.hea"))
        if "walking" not in f.parent.name.lower()
    ]
    log.info("Found %d ECG-eligible WFDB recordings.", len(hea_files))

    masterlist = []
    for hea_path in tqdm(hea_files, desc="Running CCS Nabian2018 Pipeline (Upfront 1000 Hz)"):
        session_id = hea_path.stem
        rec_path = str(hea_path.with_suffix(""))
        folder_name = hea_path.parent.name
        sub_id = session_id.split("_")[0].split("-")[0].lower()

        try:
            header = wfdb.rdheader(rec_path)
            true_fs = 1000.0 if folder_name.lower() == "sit-stand-balance" else float(header.fs)
            ecg_idx = next(
                (i for i, c in enumerate([s.upper() for s in header.sig_name]) if "ECG" in c or "EKG" in c),
                None,
            )
            if ecg_idx is None:
                continue
            epochs = extract_verified_epochs(rec_path, folder_name, header, true_fs)
        except Exception:
            log.error("Header/segmentation FAILED on %s:\n%s", session_id, traceback.format_exc())
            continue

        # Inner try/except: one bad epoch never kills remaining epochs in the same recording
        for task_name, stress_score, start_s, end_s in epochs:
            try:
                s_from = int(start_s * true_fs)
                s_to = min(header.sig_len, int(end_s * true_fs))
                if (s_to - s_from) < int(90 * true_fs):
                    continue

                raw_sig = wfdb.rdrecord(rec_path, sampfrom=s_from, sampto=s_to, channels=[ecg_idx]).p_signal[:, 0]
                ecg_raw = pd.Series(raw_sig).interpolate().bfill().ffill().values
                hrv_df = process_epoch_ccs_nabian(
                    ecg_raw, true_fs, f"{session_id}_{task_name}", sub_id, session_id, task_name, stress_score
                )
                if hrv_df is not None:
                    masterlist.append(hrv_df)
            except Exception:
                log.error("Epoch FAILED on %s_%s:\n%s", session_id, task_name, traceback.format_exc())
            finally:
                plt.close("all")
                gc.collect()

    if not masterlist:
        raise RuntimeError("No valid epochs were extracted (masterlist is empty). Check S3 sync and run_log.txt.")

    master_df = pd.concat(masterlist, axis=0, ignore_index=True)
    master_df["HR_bpm"] = 60000.0 / master_df["HRV_MeanNN"]

    id_cols = [
        "filename", "Subject", "Session", "Task", "Stress_Score", "true_srate_hz",
        "data_quality_flag", "mean_sqi", "n_rpeaks", "n_rr_total", "n_rr_bad",
        "pct_rr_bad", "is_inverted", "HR_bpm", "mean_resp_rate", "sd_resp_rate",
        "PRSA_DC_ms", "HRV_MSE_Coarse_6_15",
    ]
    master_df = master_df[id_cols + [c for c in master_df.columns if c not in id_cols]]
    master_df.to_csv(out_csv, index=False)
    log.info("Saved Nabian2018 CCS master feature table (%d epochs) to %s", len(master_df), out_csv)
    return master_df


# =============================================================================
# 3. CLINICAL PHENOTYPE MERGING, EXCLUSIONS & SUBTYPE STRATIFICATION
# =============================================================================
def prepare_analytical_cohorts(master_df: pd.DataFrame) -> tuple:
    pheno_df = pd.read_csv(CONFIG["SUBJECTS_CSV"])
    pheno_df.columns = [c.strip() for c in pheno_df.columns]

    subj_col = next((c for c in pheno_df.columns if "sub" in c.lower() or "id" in c.lower()), pheno_df.columns[0])
    master_df["Subject"] = master_df["Subject"].astype(str).str.lower().str.strip()
    pheno_df["Subject"] = pheno_df[subj_col].astype(str).str.lower().str.strip()

    df = master_df.merge(pheno_df, on="Subject", how="inner")
    df["HR_bpm"] = 60000.0 / df["HRV_MeanNN"]
    df["PRSA_DC_clean"] = df["PRSA_DC_ms"].where(df["PRSA_DC_ms"].between(0.5, 40.0), np.nan)
    df["CCS_Multiscale_Ratio"] = df["HRV_CMSEn"] / df["HRV_SampEn"].clip(lower=0.1)
    if "HRV_MSE_Coarse_6_15" in df.columns:
        df["Coarse_Multiscale_Ratio"] = df["HRV_MSE_Coarse_6_15"] / df["HRV_SampEn"].clip(lower=0.1)

    ok_df = df[df["data_quality_flag"] == "OK"].copy()
    if "completed_visit_status" in ok_df.columns:
        ok_df = ok_df[
            ~ok_df["completed_visit_status"].astype(str).str.upper().str.strip().isin(["INELIGIBLE", "EXCLUDED"])
        ].copy()

    def classify_subtype(row):
        grp = str(row.get("group", "")).upper()
        grp2 = str(row.get("group2", "")).upper()
        side = str(row.get("Stroke Side", "")).upper()
        if "STROKE" in grp or "STROKE" in grp2:
            if "RIGHT" in side or side == "R":
                return "Right MCA Stroke"
            elif "LEFT" in side or side == "L":
                return "Left MCA Stroke"
            return "Other Stroke"
        htn_col = next((c for c in row.index if "htn" in c.lower() and "history" in c.lower()), None)
        htn_val = str(row.get(htn_col, "")).upper() if htn_col else ""
        return "HTN Control" if ("HTN" in grp2 or htn_val in ["YES", "YE", "1", "1.0", "TRUE", "Y"]) else "Normotensive Control"

    ok_df["Subtype"] = ok_df.apply(classify_subtype, axis=1)

    age_col = next((c for c in ok_df.columns if c.lower() == "age"), None)
    ok_df["Age"] = pd.to_numeric(ok_df[age_col], errors="coerce") if age_col else np.nan

    bb_col = next((c for c in ok_df.columns if "BETA" in c.upper() and "BLOCK" in c.upper()), None)
    ok_df["Beta_Blocker"] = (
        ok_df[bb_col].astype(str).str.upper().str.strip().isin(["YES", "YE", "1", "1.0", "TRUE", "Y"]).astype(int)
        if bb_col else 0
    )

    sex_col = next((c for c in ok_df.columns if c.lower() in ["sex", "gender"]), None)
    ok_df["Sex_Male"] = (
        ok_df[sex_col].astype(str).str.upper().str.strip().isin(["M", "MALE", "1", "1.0"]).astype(int)
        if sex_col else np.nan
    )

    sinus_df = ok_df[ok_df["Stress_Score"].isin([0, 1])].copy()
    for af_col in [c for c in sinus_df.columns if "ATRIAL" in c.upper() or "AFIB" in c.upper()]:
        sinus_df = sinus_df[
            ~sinus_df[af_col].astype(str).str.upper().str.strip().isin(["YES", "YE", "1", "1.0", "TRUE", "Y"])
        ].copy()
    strict_sinus_ortho = sinus_df[sinus_df["HRV_RMSSD"] <= 100.0].copy()

    log.info(
        "4-State Cohort: %d subjects, %d epochs | Strict Sinus Ortho Cohort: %d subjects, %d epochs.",
        ok_df["Subject"].nunique(), len(ok_df),
        strict_sinus_ortho["Subject"].nunique(), len(strict_sinus_ortho),
    )
    return ok_df, strict_sinus_ortho


# =============================================================================
# 4. STATISTICAL MODELING & PUBLICATION CSV TABLE EXPORTS
# =============================================================================
def run_lmem_suite(sub_data: pd.DataFrame, suite_name: str, out_csv: Path) -> pd.DataFrame:
    sub_data = sub_data.copy()
    sub_data["Age_c"] = sub_data["Age"] - sub_data["Age"].mean()
    sub_data["Resp_c"] = sub_data["mean_resp_rate"] - sub_data["mean_resp_rate"].mean()
    has_bb = sub_data["Beta_Blocker"].nunique() > 1

    metrics = ["PRSA_DC_clean", "HRV_SampEn", "HRV_CMSEn", "HRV_RCMSEn", "HRV_DFA_alpha1", "CCS_Multiscale_Ratio"]
    rows = []

    for m in metrics:
        m_df = (
            sub_data.dropna(subset=[m, "Stress_Score", "Age_c", "Resp_c"])
            .groupby("Subject")
            .filter(lambda g: len(g) >= 2)
            .copy()
        )
        if len(m_df) < 10:
            continue
        cov_str = "Age_c + Beta_Blocker + Resp_c" if has_bb else "Age_c + Resp_c"
        formula = f"{m} ~ C(Subtype, Treatment(reference='Normotensive Control')) * Stress_Score + {cov_str}"
        try:
            md = smf.mixedlm(formula, data=m_df, groups=m_df["Subject"]).fit(method=["powell", "nm"], reml=True)
            var_u0 = float(md.cov_re.iloc[0, 0]) if md.cov_re.shape[0] > 0 else 0.0
            icc = var_u0 / (var_u0 + md.scale) if (var_u0 + md.scale) > 0 else 0.0
            ci = md.conf_int()
            for param in md.fe_params.index:
                rows.append({
                    "Model_Suite": suite_name,
                    "Outcome": m,
                    "N_Epochs": len(m_df),
                    "N_Subjects": m_df["Subject"].nunique(),
                    "Parameter": param,
                    "Beta": round(float(md.fe_params[param]), 4),
                    "SE": round(float(md.bse_fe[param]), 4),
                    "z_stat": round(float(md.tvalues[param]), 3),
                    "p_value": round(float(md.pvalues[param]), 5),
                    "CI_95_Low": round(float(ci.loc[param, 0]), 4),
                    "CI_95_High": round(float(ci.loc[param, 1]), 4),
                    "ICC": round(icc, 4),
                })
        except Exception as exc:
            log.warning("LMEM fallback to GEE for %s (%s): %s", m, suite_name, exc)
            gee = smf.gee(formula, groups=m_df["Subject"], data=m_df, cov_struct=sm.cov_struct.Exchangeable()).fit()
            ci = gee.conf_int()
            for param in gee.params.index:
                rows.append({
                    "Model_Suite": f"{suite_name}_GEE_Fallback",
                    "Outcome": m,
                    "N_Epochs": len(m_df),
                    "N_Subjects": m_df["Subject"].nunique(),
                    "Parameter": param,
                    "Beta": round(float(gee.params[param]), 4),
                    "SE": round(float(gee.bse[param]), 4),
                    "z_stat": round(float(gee.tvalues[param]), 3),
                    "p_value": round(float(gee.pvalues[param]), 5),
                    "CI_95_Low": round(float(ci.loc[param, 0]), 4),
                    "CI_95_High": round(float(ci.loc[param, 1]), 4),
                    "ICC": np.nan,
                })

    res_df = pd.DataFrame(rows)
    res_df.to_csv(out_csv, index=False)
    log.info("Exported LMEM table (%s) -> %s", suite_name, out_csv)
    return res_df


def extract_eb_slope(sub_data: pd.DataFrame, metric_name: str, label: str) -> pd.DataFrame:
    d = (
        sub_data.dropna(subset=[metric_name, "Stress_Score"])
        .groupby("Subject")
        .filter(lambda g: len(g) >= 2)
        .copy()
    )
    if d.empty:
        return pd.DataFrame(columns=["Subject", label])
    try:
        fit = smf.mixedlm(
            f"{metric_name} ~ Stress_Score", data=d, groups=d["Subject"], re_formula="~Stress_Score"
        ).fit(method=["powell", "nm"], reml=True)
        return pd.DataFrame([
            {"Subject": s, label: float(v.iloc[1]) if len(v) > 1 else np.nan}
            for s, v in fit.random_effects.items()
        ])
    except Exception:
        return pd.DataFrame({"Subject": d["Subject"].unique(), label: np.nan})


def export_all_tables(ok_df: pd.DataFrame, strict_sinus_ortho: pd.DataFrame) -> tuple:
    metrics = [
        "HR_bpm", "mean_resp_rate", "HRV_SDNN", "HRV_RMSSD", "PRSA_DC_clean",
        "HRV_SampEn", "HRV_DFA_alpha1", "HRV_CMSEn", "HRV_RCMSEn", "CCS_Multiscale_Ratio",
    ]
    if "Coarse_Multiscale_Ratio" in ok_df.columns:
        metrics.append("Coarse_Multiscale_Ratio")

    t_task = ok_df.groupby(["Stress_Score", "Task"])[metrics].agg(["count", "mean", "std", "median"]).round(3)
    t_task.columns = [f"{col}_{stat}" for col, stat in t_task.columns]
    t_task.reset_index().to_csv(TABLE_DIR / "Table2_Task_State_Descriptives.csv", index=False)

    rest_cols = ["Age", "Beta_Blocker", "Sex_Male", "PRSA_DC_clean", "HRV_SampEn", "HRV_CMSEn", "HRV_RCMSEn", "HRV_DFA_alpha1", "CCS_Multiscale_Ratio"]
    if "Coarse_Multiscale_Ratio" in ok_df.columns:
        rest_cols.append("Coarse_Multiscale_Ratio")

    rest_subj = (
        ok_df[ok_df["Stress_Score"] == 0]
        .groupby(["Subject", "Subtype"], as_index=False)[rest_cols]
        .mean()
    )
    demo_rows = []
    for st, grp in rest_subj.groupby("Subtype"):
        row = {
            "Subtype": st,
            "n_Subjects": len(grp),
            "Age_Mean": round(grp["Age"].mean(), 2),
            "Age_SD": round(grp["Age"].std(), 2),
            "Male_n": int(grp["Sex_Male"].sum()) if grp["Sex_Male"].notna().any() else np.nan,
            "Beta_Blocker_n": int(grp["Beta_Blocker"].sum()),
            "PRSA_DC_Rest_Mean": round(grp["PRSA_DC_clean"].mean(), 3),
            "PRSA_DC_Rest_SD": round(grp["PRSA_DC_clean"].std(), 3),
            "SampEn_Rest_Mean": round(grp["HRV_SampEn"].mean(), 3),
            "SampEn_Rest_SD": round(grp["HRV_SampEn"].std(), 3),
            "CCS_Ratio_Rest_Mean": round(grp["CCS_Multiscale_Ratio"].mean(), 3),
            "CCS_Ratio_Rest_SD": round(grp["CCS_Multiscale_Ratio"].std(), 3),
        }
        if "Coarse_Multiscale_Ratio" in grp.columns:
            row["Coarse_Ratio_Rest_Mean"] = round(grp["Coarse_Multiscale_Ratio"].mean(), 3)
            row["Coarse_Ratio_Rest_SD"] = round(grp["Coarse_Multiscale_Ratio"].std(), 3)
        demo_rows.append(row)
    pd.DataFrame(demo_rows).to_csv(TABLE_DIR / "Table1_Demographics_and_Resting_Subtypes.csv", index=False)

    ntn = rest_subj[rest_subj["Subtype"] == "Normotensive Control"]
    htn = rest_subj[rest_subj["Subtype"] == "HTN Control"]
    lmca = rest_subj[rest_subj["Subtype"] == "Left MCA Stroke"]
    rmca = rest_subj[rest_subj["Subtype"] == "Right MCA Stroke"]
    all_stroke = rest_subj[rest_subj["Subtype"].str.contains("Stroke")]

    t_dc, p_dc = stats.ttest_ind(ntn["PRSA_DC_clean"].dropna(), all_stroke["PRSA_DC_clean"].dropna(), equal_var=False)
    u_dc, pu_dc = stats.mannwhitneyu(ntn["PRSA_DC_clean"].dropna(), all_stroke["PRSA_DC_clean"].dropna())
    f_rat, pf_rat = stats.f_oneway(
        ntn["CCS_Multiscale_Ratio"].dropna(),
        htn["CCS_Multiscale_Ratio"].dropna(),
        lmca["CCS_Multiscale_Ratio"].dropna(),
        rmca["CCS_Multiscale_Ratio"].dropna(),
    )
    t_rl, p_rl = stats.ttest_ind(rmca["CCS_Multiscale_Ratio"].dropna(), lmca["CCS_Multiscale_Ratio"].dropna(), equal_var=False)

    hyp_rows = [
        {"Hypothesis_Test": "H1: 4-Group ANOVA (CCS_Multiscale_Ratio)", "Statistic": round(f_rat, 3), "p_value": round(pf_rat, 5)},
        {"Hypothesis_Test": "H1: Right MCA vs Left MCA (CCS_Multiscale_Ratio, Welch t)", "Statistic": round(t_rl, 3), "p_value": round(p_rl, 5)},
        {"Hypothesis_Test": "H2: Normotensive vs All Stroke (PRSA_DC_clean, Welch t)", "Statistic": round(t_dc, 3), "p_value": round(p_dc, 5)},
        {"Hypothesis_Test": "H2: Normotensive vs All Stroke (PRSA_DC_clean, Mann-Whitney U)", "Statistic": round(u_dc, 3), "p_value": round(pu_dc, 5)},
    ]
    if "Coarse_Multiscale_Ratio" in rest_subj.columns:
        f_cr, pf_cr = stats.f_oneway(
            ntn["Coarse_Multiscale_Ratio"].dropna(),
            htn["Coarse_Multiscale_Ratio"].dropna(),
            lmca["Coarse_Multiscale_Ratio"].dropna(),
            rmca["Coarse_Multiscale_Ratio"].dropna(),
        )
        t_crl, p_crl = stats.ttest_ind(rmca["Coarse_Multiscale_Ratio"].dropna(), lmca["Coarse_Multiscale_Ratio"].dropna(), equal_var=False)
        hyp_rows.append({"Hypothesis_Test": "H1: 4-Group ANOVA (Coarse_Multiscale_Ratio)", "Statistic": round(f_cr, 3), "p_value": round(pf_cr, 5)})
        hyp_rows.append({"Hypothesis_Test": "H1: Right MCA vs Left MCA (Coarse_Multiscale_Ratio, Welch t)", "Statistic": round(t_crl, 3), "p_value": round(p_crl, 5)})

    pd.DataFrame(hyp_rows).to_csv(TABLE_DIR / "Table1B_Resting_Hypothesis_Tests.csv", index=False)

    run_lmem_suite(strict_sinus_ortho, "Strict_Sinus_Orthostatic_0to1", TABLE_DIR / "Table3_LMEM_StrictSinus_Orthostatic.csv")
    run_lmem_suite(ok_df, "Calibrated_4State_Continuum_0to3", TABLE_DIR / "Table4_LMEM_4State_Continuum.csv")

    ortho_all = ok_df[ok_df["Stress_Score"].isin([0, 1])].copy()
    tilt_only = ok_df[ok_df["Task"].isin(["State0_SupineRest", "State1_HeadUpTilt"])].copy()
    stand_only = ok_df[ok_df["Task"].isin(["State0_SeatedRest", "State1_SitToStand"])].copy()

    bio = rest_subj[["Subject", "PRSA_DC_clean", "CCS_Multiscale_Ratio", "HRV_SampEn", "HRV_RCMSEn", "HRV_DFA_alpha1"]].rename(
        columns={c: f"{c}_Rest" for c in ["PRSA_DC_clean", "CCS_Multiscale_Ratio", "HRV_SampEn", "HRV_RCMSEn", "HRV_DFA_alpha1"]}
    )
    for m_col, short_name in [("HRV_DFA_alpha1", "DFAa1"), ("HRV_SampEn", "SampEn"), ("HRV_RCMSEn", "RCMSEn"), ("PRSA_DC_clean", "DC")]:
        bio = bio.merge(extract_eb_slope(ok_df, m_col, f"u1i_4State_{short_name}"), on="Subject", how="left")
        bio = bio.merge(extract_eb_slope(ortho_all, m_col, f"u1i_Ortho_{short_name}"), on="Subject", how="left")
        bio = bio.merge(extract_eb_slope(tilt_only, m_col, f"u1i_Day2Tilt_{short_name}"), on="Subject", how="left")
        bio = bio.merge(extract_eb_slope(stand_only, m_col, f"u1i_Day1Stand_{short_name}"), on="Subject", how="left")

    subj_master = ok_df.drop_duplicates("Subject").merge(bio, on="Subject", how="inner")
    endpoints = [
        "DELTA MEAN MCAL TILT-BASELINE", "DELTA MEAN MCAR TILT-BASELINE",
        "MMPF_Tilt_Mean_PhaseShift(Left)", "MMPF_Tilt_Mean_PhaseShift(Right)",
        "TM: Part B Time (sec)", "GAIT - Walk 2 speed (m/s)", "GAIT - Walk 1 speed (m/s)", "NIHSS",
    ]
    bio_cols = [c for c in bio.columns if c != "Subject"]
    corr_rows = []
    for ep in endpoints:
        if ep in subj_master.columns:
            for b in bio_cols:
                v = subj_master[[b, ep]].apply(pd.to_numeric, errors="coerce").dropna()
                if len(v) >= 20:
                    r_val, p_val = stats.spearmanr(v[b], v[ep])
                    corr_rows.append({
                        "Clinical_Endpoint": ep,
                        "HRV_Biomarker": b,
                        "N": len(v),
                        "Spearman_rs": round(float(r_val), 4),
                        "R2_Rank_Pct": round(float((r_val ** 2) * 100.0), 2),
                        "p_value": round(float(p_val), 5),
                    })
    if corr_rows:
        corr_df = pd.DataFrame(corr_rows).sort_values(["Clinical_Endpoint", "p_value"])
        corr_df.to_csv(TABLE_DIR / "Table5_Translational_Spearman_Coupling.csv", index=False)
    log.info("All publication CSV tables exported successfully.")
    return rest_subj, subj_master


# =============================================================================
# 5. PUBLICATION & SUPPLEMENTARY FIGURES (SEABORN >= 0.14 SAFE)
# =============================================================================
def generate_figure2_hemispheric_dissociation(rest_subj: pd.DataFrame) -> None:
    sns.set_theme(style="whitegrid", font_scale=1.05)
    order = ["Normotensive Control", "HTN Control", "Left MCA Stroke", "Right MCA Stroke", "Other Stroke"]
    palette = {
        "Normotensive Control": "#1b9e77",
        "HTN Control": "#386cb0",
        "Left MCA Stroke": "#d95f02",
        "Right MCA Stroke": "#d62728",
        "Other Stroke": "#7570b3",
    }

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 6.0), dpi=300)

    # Panel A (hue="Subtype", legend=False required for Seaborn >= 0.14)
    sns.boxplot(
        data=rest_subj, x="Subtype", y="CCS_Multiscale_Ratio", hue="Subtype",
        order=order, hue_order=order, palette=palette, legend=False,
        width=0.52, showfliers=False, boxprops={"alpha": 0.45}, ax=axes[0],
    )
    sns.stripplot(
        data=rest_subj, x="Subtype", y="CCS_Multiscale_Ratio", hue="Subtype",
        order=order, hue_order=order, palette=palette, legend=False,
        size=6.5, alpha=0.85, jitter=0.18, ax=axes[0],
    )
    axes[0].axhline(1.0, color="#111827", linestyle="--", lw=1.4, label="Scale-Invariant Threshold (1.0)")
    axes[0].set_title("A. Hemispheric Dissociation in Resting Multiscale Complexity Ratio", fontweight="bold")
    axes[0].set_ylabel("Resting CCS Multiscale Ratio ($CMSEn / SampEn$)", fontweight="bold")
    axes[0].set_xlabel("")
    axes[0].tick_params(axis="x", rotation=18)
    axes[0].legend(loc="upper right", frameon=True)

    # Panel B
    sns.boxplot(
        data=rest_subj, x="Subtype", y="PRSA_DC_clean", hue="Subtype",
        order=order, hue_order=order, palette=palette, legend=False,
        width=0.52, showfliers=False, boxprops={"alpha": 0.45}, ax=axes[1],
    )
    sns.stripplot(
        data=rest_subj, x="Subtype", y="PRSA_DC_clean", hue="Subtype",
        order=order, hue_order=order, palette=palette, legend=False,
        size=6.5, alpha=0.85, jitter=0.18, ax=axes[1],
    )
    axes[1].axhline(4.5, color="#9B2226", linestyle="--", lw=1.4, label="Clinical Risk Threshold (4.5 ms)")
    axes[1].set_title("B. Pan-Stroke Depletion of Resting Cardiovagal Reserve (PRSA DC)", fontweight="bold")
    axes[1].set_ylabel("Resting PRSA Deceleration Capacity (ms)", fontweight="bold")
    axes[1].set_xlabel("")
    axes[1].tick_params(axis="x", rotation=18)
    axes[1].legend(loc="upper right", frameon=True)

    sns.despine()
    plt.tight_layout()
    for ext in ["png", "pdf"]:
        plt.savefig(FIG_DIR / f"Figure2_Hemispheric_Dissociation_PRSA_DC.{ext}", dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def generate_figure3_translational_4panel(subj_master: pd.DataFrame) -> None:
    sns.set_theme(style="whitegrid", font_scale=1.05)
    fig, axes = plt.subplots(2, 2, figsize=(14, 11), dpi=300)

    phase_col = "MMPF_Tilt_Mean_PhaseShift(Right)" if "MMPF_Tilt_Mean_PhaseShift(Right)" in subj_master.columns else "MMPF_Tilt_Mean_PhaseShift(Left)"
    panel_specs = [
        (axes[0, 0], "CCS_Multiscale_Ratio_Rest", "DELTA MEAN MCAL TILT-BASELINE", "#1b9e77", "#d62728",
         "A. Multiscale Complexity Ratio vs. Orthostatic CBFV", "Resting CCS Multiscale Ratio ($CMSEn / SampEn$)", "$\\Delta$ Mean Left MCA CBFV on Tilt (cm/s)"),
        (axes[0, 1], "PRSA_DC_clean_Rest", phase_col, "#2b5c8f", "#d62728",
         "B. Cardiovagal Reserve ($DC$) vs. Cerebral Phase Shift", "Resting PRSA Deceleration Capacity ($DC$, ms)", "Tilt MMPF Pressure-Flow Phase Shift ($\\Delta\\theta$, deg)"),
        (axes[1, 0], "PRSA_DC_clean_Rest", "TM: Part B Time (sec)", "#7570b3", "#d62728",
         "C. Cardiovagal Reserve ($DC$) vs. Executive Function", "Resting PRSA Deceleration Capacity ($DC$, ms)", "Trail Making Test Part B Completion Time (sec)"),
        (axes[1, 1], "u1i_Ortho_DFAa1", "GAIT - Walk 2 speed (m/s)", "#d95f02", "#2b5c8f",
         "D. Orthostatic Reactivity ($u_{1i}$) vs. Hallway Gait Speed", "Orthostatic DFA $\\alpha_1$ Reactivity Slope ($u_{1i}$)", "12-Min Hallway Walking Speed — Walk 2 (m/s)"),
    ]

    for ax, x_col, y_col, sc_col, ln_col, title_prefix, xlabel, ylabel in panel_specs:
        if x_col not in subj_master.columns or y_col not in subj_master.columns:
            ax.text(0.5, 0.5, f"Missing column:\n{x_col} or {y_col}", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(title_prefix, fontweight="bold")
            continue
        v = subj_master[[x_col, y_col]].apply(pd.to_numeric, errors="coerce").dropna()
        if len(v) < 3:
            ax.text(0.5, 0.5, f"Insufficient paired data (n={len(v)})", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(title_prefix, fontweight="bold")
            continue
        r_val, p_val = stats.spearmanr(v[x_col], v[y_col])
        sns.regplot(data=v, x=x_col, y=y_col, scatter_kws={"alpha": 0.78, "s": 55, "color": sc_col}, line_kws={"color": ln_col, "lw": 2.2}, ax=ax)
        ax.set_title(f"{title_prefix}\n(Spearman $r_s = {r_val:.3f}, p = {p_val:.4f}, n = {len(v)}$)", fontweight="bold")
        ax.set_xlabel(xlabel, fontweight="bold")
        ax.set_ylabel(ylabel, fontweight="bold")

    sns.despine()
    plt.tight_layout()
    for ext in ["png", "pdf"]:
        plt.savefig(FIG_DIR / f"Figure3_Translational_4Panel_Synthesis.{ext}", dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def generate_supplementary_figure_s1(ok_df: pd.DataFrame) -> None:
    sns.set_theme(style="whitegrid", font_scale=1.0)
    palette = {
        "Normotensive Control": "#1b9e77",
        "HTN Control": "#386cb0",
        "Left MCA Stroke": "#d95f02",
        "Right MCA Stroke": "#d62728",
        "Other Stroke": "#7570b3",
    }
    label_map = {0: "State 0\nRest", 1: "State 1\nTilt/Stand", 2: "State 2\nBalance EO", 3: "State 3\nBalance EC"}
    present_states = sorted(int(s) for s in ok_df["Stress_Score"].dropna().unique())
    tick_labels = [label_map.get(s, f"State {s}") for s in present_states]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=300)
    specs = [
        ("PRSA_DC_clean", "A. PRSA Deceleration Capacity (ms)", axes[0, 0]),
        ("HRV_SampEn", "B. Single-Scale Sample Entropy (SampEn)", axes[0, 1]),
        ("HRV_CMSEn", "C. Composite Multiscale Entropy (CMSEn)", axes[1, 0]),
        ("HRV_DFA_alpha1", "D. Short-Term Fractal Scaling (DFA α1)", axes[1, 1]),
    ]
    for col, title, ax in specs:
        sns.pointplot(
            data=ok_df, x="Stress_Score", y=col, hue="Subtype",
            order=present_states, palette=palette, errorbar="se", markers="o", dodge=0.25, capsize=0.06, ax=ax,
        )
        ax.set_title(title, fontweight="bold")
        ax.set_xticks(range(len(present_states)))
        ax.set_xticklabels(tick_labels)
        ax.set_xlabel("")
        ax.set_ylabel(col, fontweight="bold")
        if col != "PRSA_DC_clean" and ax.get_legend() is not None:
            ax.get_legend().remove()
        elif col == "PRSA_DC_clean":
            ax.legend(title="Clinical Subtype", fontsize=8.5, title_fontsize=9, loc="upper right")

    sns.despine()
    plt.tight_layout()
    for ext in ["png", "pdf"]:
        plt.savefig(SUPP_FIG_DIR / f"FigureS1_4State_Continuum_Trajectories.{ext}", dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)

def export_vif_and_lmem_diagnostics(ok_df: pd.DataFrame, strict_sinus_ortho: pd.DataFrame) -> dict:
    """Generates Table_S2_VIF_Multicollinearity.csv, Table_S3_LMEM_Diagnostics.csv, and returns fitted 4-state models."""
    lmem_df = ok_df.groupby("Subject").filter(lambda g: len(g) >= 2).copy()
    lmem_df["Age_c"] = lmem_df["Age"] - lmem_df["Age"].mean()
    lmem_df["Resp_c"] = lmem_df["mean_resp_rate"] - lmem_df["mean_resp_rate"].mean()

    ortho_df = strict_sinus_ortho.groupby("Subject").filter(lambda g: len(g) >= 2).copy()
    ortho_df["Age_c"] = ortho_df["Age"] - ortho_df["Age"].mean()
    ortho_df["Resp_c"] = ortho_df["mean_resp_rate"] - ortho_df["mean_resp_rate"].mean()

    # 1. VIF Multicollinearity Table (Table S2)
    vif_sub = lmem_df.dropna(subset=["HRV_SampEn", "Stress_Score", "Age_c", "Beta_Blocker", "Resp_c"]).copy()
    has_bb = vif_sub["Beta_Blocker"].nunique() > 1
    cov_str = "Age_c + Beta_Blocker + Resp_c" if has_bb else "Age_c + Resp_c"
    _, X_vif = patsy.dmatrices(
        f"HRV_SampEn ~ C(Subtype, Treatment(reference='Normotensive Control')) + Stress_Score + {cov_str}",
        data=vif_sub, return_type="dataframe"
    )
    vif_rows = []
    for i, col in enumerate(X_vif.columns):
        if col != "Intercept":
            v = float(variance_inflation_factor(X_vif.values, i))
            vif_rows.append({
                "Predictor": col.replace("C(Subtype, Treatment(reference='Normotensive Control'))", "Subtype"),
                "VIF": round(v, 3),
                "Tolerance": round(1.0 / v, 3) if v > 0 else np.nan,
                "Status": "Pass (VIF < 5.0)" if v < 5.0 else "Check Collinearity",
            })
    pd.DataFrame(vif_rows).to_csv(TABLE_DIR / "Table_S2_VIF_Multicollinearity.csv", index=False)

    # 2. Variance Components, ICC, Marginal/Conditional R2 & Shapiro-Wilk (Table S3)
    diag_rows = []
    models_for_plot = {}
    metrics = ["PRSA_DC_clean", "HRV_SampEn", "HRV_CMSEn", "HRV_RCMSEn", "HRV_DFA_alpha1", "CCS_Multiscale_Ratio"]

    for suite_label, dataset in [("Clean_Sinus_Ortho_0to1", ortho_df), ("Calibrated_4State_0to3", lmem_df)]:
        suite_bb = dataset["Beta_Blocker"].nunique() > 1
        suite_cov = "Age_c + Beta_Blocker + Resp_c" if suite_bb else "Age_c + Resp_c"
        for m in metrics:
            sub_m = dataset.dropna(subset=[m, "Stress_Score", "Age_c", "Resp_c"]).groupby("Subject").filter(lambda g: len(g) >= 2).copy()
            if len(sub_m) < 10:
                continue
            formula = f"{m} ~ C(Subtype, Treatment(reference='Normotensive Control')) * Stress_Score + {suite_cov}"
            try:
                md = smf.mixedlm(formula, data=sub_m, groups=sub_m["Subject"]).fit(method=["powell", "nm"], reml=True)
                var_u0 = float(md.cov_re.iloc[0, 0]) if md.cov_re.shape[0] > 0 else 0.0
                var_eps = float(md.scale)
                icc = var_u0 / (var_u0 + var_eps) if (var_u0 + var_eps) > 0 else 0.0
                var_fixed = float(np.var(np.dot(md.model.exog, md.fe_params), ddof=1))
                tot_var = var_fixed + var_u0 + var_eps
                w_stat, w_p = stats.shapiro(md.resid)

                diag_rows.append({
                    "Model_Suite": suite_label,
                    "Outcome": m,
                    "N_Epochs": len(sub_m),
                    "N_Subjects": sub_m["Subject"].nunique(),
                    "Var_u0": round(var_u0, 4),
                    "Var_eps": round(var_eps, 4),
                    "ICC": round(icc, 4),
                    "R2_Marginal": round(var_fixed / tot_var, 4) if tot_var > 0 else np.nan,
                    "R2_Conditional": round((var_fixed + var_u0) / tot_var, 4) if tot_var > 0 else np.nan,
                    "REML_LogLik": round(float(md.llf), 2),
                    "Shapiro_W": round(float(w_stat), 4),
                    "Shapiro_p": round(float(w_p), 5),
                })
                if suite_label == "Calibrated_4State_0to3" and m in ["PRSA_DC_clean", "HRV_SampEn", "HRV_CMSEn", "HRV_DFA_alpha1"]:
                    models_for_plot[m] = md
            except Exception as exc:
                log.warning("Diagnostics skipped for %s (%s): %s", m, suite_label, exc)

    pd.DataFrame(diag_rows).to_csv(TABLE_DIR / "Table_S3_LMEM_Diagnostics_ICC_R2_Shapiro.csv", index=False)
    log.info("Exported Table S2 (VIF) and Table S3 (LMEM Diagnostics).")
    return models_for_plot


def generate_supplementary_figure_s2_lmem_diagnostics(models_for_plot: dict) -> None:
    """Generates 2x4 Q-Q and Residuals-vs-Fitted diagnostic plots."""
    outcomes = [
        ("PRSA_DC_clean", "PRSA DC (ms)"),
        ("HRV_SampEn", "Sample Entropy"),
        ("HRV_CMSEn", "Composite MSE"),
        ("HRV_DFA_alpha1", "DFA α1"),
    ]
    sns.set_theme(style="whitegrid", font_scale=1.0)
    fig, axes = plt.subplots(2, 4, figsize=(16, 8), dpi=300)

    for idx, (col_key, label) in enumerate(outcomes):
        if col_key not in models_for_plot:
            continue
        md = models_for_plot[col_key]
        res = md.resid
        std_res = (res - np.mean(res)) / np.std(res, ddof=1)

        (osm, osr), (slope, intercept, r_qq) = stats.probplot(std_res, dist="norm")
        axes[0, idx].scatter(osm, osr, alpha=0.65, s=26, color="#2b5c8f", edgecolor="none")
        axes[0, idx].plot(osm, slope * osm + intercept, color="#d62728", lw=2.0)
        axes[0, idx].set_title(f"{chr(65+idx)}1. Q-Q: {label} (r = {r_qq:.3f})", fontweight="bold", fontsize=10)
        axes[0, idx].set_xlabel("Theoretical Quantiles")
        axes[0, idx].set_ylabel("Standardized Residuals")

        axes[1, idx].scatter(md.fittedvalues, std_res, alpha=0.65, s=26, color="#1b9e77", edgecolor="none")
        axes[1, idx].axhline(0.0, color="#d62728", linestyle="--", lw=1.5)
        axes[1, idx].axhline(2.5, color="gray", linestyle=":", lw=1.0)
        axes[1, idx].axhline(-2.5, color="gray", linestyle=":", lw=1.0)
        axes[1, idx].set_title(f"{chr(65+idx)}2. Residuals vs. Fitted", fontweight="bold", fontsize=10)
        axes[1, idx].set_xlabel("REML Fitted Values")
        axes[1, idx].set_ylabel("Standardized Residuals")

    sns.despine()
    plt.tight_layout()
    for ext in ["png", "pdf"]:
        plt.savefig(SUPP_FIG_DIR / f"FigureS2_LMEM_Residual_Diagnostics.{ext}", dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def generate_supplementary_figure_s3_forensic_gait(ok_df: pd.DataFrame, subj_master: pd.DataFrame) -> None:
    """Self-contained forensic proof: 1000 Hz clock calibration + 4-State vs Pure Ortho gait slope reversal."""
    sns.set_theme(style="whitegrid", font_scale=1.0)
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), dpi=300)

    # Panel A: Calibrated (1000 Hz) vs Simulated Uncalibrated (500 Hz header bug = 2x ms in States 2 & 3)
    cal = ok_df[["Stress_Score", "PRSA_DC_clean"]].copy()
    cal["Pipeline"] = "1000 Hz Calibrated (True LabVIEW Clock)"
    uncal = ok_df[["Stress_Score", "PRSA_DC_clean"]].copy()
    uncal.loc[uncal["Stress_Score"].isin([2, 3]), "PRSA_DC_clean"] *= 2.0
    uncal["Pipeline"] = "Uncalibrated 500 Hz (.hea Header Bug)"
    comb = pd.concat([cal, uncal], ignore_index=True)

    sns.pointplot(
        data=comb, x="Stress_Score", y="PRSA_DC_clean", hue="Pipeline",
        palette={"1000 Hz Calibrated (True LabVIEW Clock)": "#1b9e77", "Uncalibrated 500 Hz (.hea Header Bug)": "#d62728"},
        dodge=0.15, markers=["o", "X"], capsize=0.08, err_kws={"linewidth": 1.4}, ax=axes[0],
    )
    axes[0].set_title("A. 1000 Hz Header Calibration in PRSA DC", fontweight="bold", fontsize=10.5)
    axes[0].set_xlabel("Protocol State (0=Rest, 1=Ortho, 2=Bal EO, 3=Bal EC)", fontweight="bold")
    axes[0].set_ylabel("PRSA Deceleration Capacity (ms)", fontweight="bold")
    axes[0].legend(fontsize=8.5, loc="upper left")

    # Panels B & C: 4-State vs Pure Orthostatic DFAa1 Slope against Walk 2 Speed
    gait_col = "GAIT - Walk 2 speed (m/s)"
    for ax, slope_col, color_pt, color_ln, title_txt, xlabel_txt in [
        (axes[1], "u1i_4State_DFAa1", "#7570b3", "#d62728", "B. Balance-Confounded 4-State DFA α1 Slope", "4-State DFA α1 Slope (u1i, States 0→3)"),
        (axes[2], "u1i_Ortho_DFAa1", "#d95f02", "#2b5c8f", "C. Pure Orthostatic Baroreflex DFA α1 Slope", "Pure Orthostatic DFA α1 Slope (u1i, States 0→1)"),
    ]:
        if slope_col in subj_master.columns and gait_col in subj_master.columns:
            v = subj_master[[slope_col, gait_col]].apply(pd.to_numeric, errors="coerce").dropna()
            if len(v) >= 3:
                r_val, p_val = stats.spearmanr(v[slope_col], v[gait_col])
                sns.regplot(data=v, x=slope_col, y=gait_col, scatter_kws={"alpha": 0.75, "s": 45, "color": color_pt}, line_kws={"color": color_ln, "lw": 2.0}, ax=ax)
                ax.set_title(f"{title_txt}\n(Spearman $r_s = {r_val:+.3f}, p = {p_val:.4f}, N = {len(v)}$)", fontweight="bold", fontsize=10.5)
        ax.set_xlabel(xlabel_txt, fontweight="bold")
        ax.set_ylabel("12-Min Hallway Walking Speed — Walk 2 (m/s)", fontweight="bold")

    sns.despine()
    plt.tight_layout()
    for ext in ["png", "pdf"]:
        plt.savefig(SUPP_FIG_DIR / f"FigureS3_Forensic_Calibration_Gait.{ext}", dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
# =============================================================================
# 6. SAFE PACKAGING (LIGHTWEIGHT PUBLISHER ZIP + OPTIONAL DRIVE BACKUP)
# =============================================================================
def package_and_download_zip() -> Path:
    """
    Packages Tables, Publication Figures, Supplementary Figures, and run_log.txt into
    a lightweight ZIP (< 20 MB) so Colab browser downloads never time out.
    If Google Drive is mounted at /content/drive/MyDrive, copies the full archive there.
    """
    zip_path = ROOT_WORK / CONFIG["ZIP_FILENAME"]
    include_dirs = [FIG_DIR, SUPP_FIG_DIR, TABLE_DIR]
    log.info("Packaging lightweight Publisher ZIP into %s...", zip_path)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for d in include_dirs:
            for root, _, filenames in os.walk(d):
                for fname in filenames:
                    fpath = Path(root) / fname
                    zipf.write(fpath, arcname=str(fpath.relative_to(OUT_DIR)))
        run_log = OUT_DIR / "run_log.txt"
        if run_log.exists():
            zipf.write(run_log, arcname="run_log.txt")

    size_mb = zip_path.stat().st_size / 1e6
    log.info("Lightweight Publisher ZIP ready: %s (%.2f MB)", zip_path, size_mb)

    # Auto-backup to Google Drive if already mounted
    gdrive_dir = Path("/content/drive/MyDrive")
    if IS_COLAB and gdrive_dir.exists():
        dest = gdrive_dir / CONFIG["ZIP_FILENAME"]
        shutil.copy2(zip_path, dest)
        log.info("Copied Publisher ZIP to Google Drive: %s", dest)

    if IS_COLAB and CONFIG["TRIGGER_COLAB_DOWNLOAD"] and size_mb < 200.0:
        try:
            from google.colab import files
            files.download(str(zip_path))
        except Exception as exc:
            log.warning("Colab browser download skipped: %s", exc)
    return zip_path


# =============================================================================
# MAIN EXECUTION ENTRYPOINT
# =============================================================================
def main() -> None:
    if CONFIG["DOWNLOAD_DATA"]:
        download_cves_dataset()

    master_csv = TABLE_DIR / "Table_S1_Master_Epoch_Features_Nabian2018_1000Hz.csv"
    if CONFIG["RUN_EXTRACTION"] or not master_csv.exists():
        master_df = run_extraction_pipeline()
    else:
        log.info("Loading existing Nabian2018 CCS master CSV: %s", master_csv)
        master_df = pd.read_csv(master_csv)

    ok_df, strict_sinus_ortho = prepare_analytical_cohorts(master_df)
    rest_subj, subj_master = export_all_tables(ok_df, strict_sinus_ortho)

    generate_figure2_hemispheric_dissociation(rest_subj)
    generate_figure3_translational_4panel(subj_master)
    generate_supplementary_figure_s1(ok_df)
    models_for_plot = export_vif_and_lmem_diagnostics(ok_df, strict_sinus_ortho)
    generate_supplementary_figure_s2_lmem_diagnostics(models_for_plot)
    generate_supplementary_figure_s3_forensic_gait(ok_df, subj_master)
    package_and_download_zip()


if __name__ == "__main__":
    main()

18:09:13  [INFO]  Environment: Kaggle | Workspace: /kaggle/working/cves_nabian2018_ccs_workspace
18:09:13  [INFO]  Loading existing Nabian2018 CCS master CSV: /kaggle/working/cves_nabian2018_ccs_workspace/Tables_CSV/Table_S1_Master_Epoch_Features_Nabian2018_1000Hz.csv
18:09:13  [INFO]  4-State Cohort: 72 subjects, 386 epochs | Strict Sinus Ortho Cohort: 68 subjects, 237 epochs.
18:09:15  [INFO]  Exported LMEM table (Strict_Sinus_Orthostatic_0to1) -> /kaggle/working/cves_nabian2018_ccs_workspace/Tables_CSV/Table3_LMEM_StrictSinus_Orthostatic.csv
18:09:17  [INFO]  Exported LMEM table (Calibrated_4State_Continuum_0to3) -> /kaggle/working/cves_nabian2018_ccs_workspace/Tables_CSV/Table4_LMEM_4State_Continuum.csv
18:09:34  [INFO]  All publication CSV tables exported successfully.
18:09:40  [INFO]  Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
18:09:4

In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Kaggle Output Packager for GitHub Upload
Tailored to the exact /kaggle/working directory structure:
    /kaggle/working/
    ├── cves_nabian2018_ccs_workspace/
    │   ├── Figures_Publication/
    │   ├── Figures_Supplementary/
    │   ├── Supplementary_QC_Plots/
    │   ├── Supplementary_Tachograms/
    │   ├── Tables_CSV/
    │   └── run_log.txt
    ├── cves-dataset/          (EXCLUDED - raw PhysioNet data)
    └── .virtual_documents/    (EXCLUDED - Kaggle internal files)

Creates two clean, relative-path ZIP archives in /kaggle/working/:
    1. CVES_Full_Workspace_For_Git.zip  (All 5 subfolders + run_log.txt + .gitignore)
    2. CVES_Light_For_Git.zip           (Tables, Publication/Supp Figures, Tachograms, log — excludes QC plots)

Displays clickable IPython FileLinks for immediate local download in Kaggle.
"""

import os
import zipfile
from pathlib import Path

# =============================================================================
# 1. PATH CONFIGURATION (MATCHED TO /kaggle/working STRUCTURE)
# =============================================================================
WORKING_DIR = Path("/kaggle/working")
WORKSPACE_DIR = WORKING_DIR / "cves_nabian2018_ccs_workspace"

# Expected subfolders and files inside cves_nabian2018_ccs_workspace
EXPECTED_SUBDIRS = [
    "Figures_Publication",
    "Figures_Supplementary",
    "Supplementary_QC_Plots",
    "Supplementary_Tachograms",
    "Tables_CSV",
]
EXPECTED_FILES = ["run_log.txt"]

# Output ZIP paths
FULL_ZIP_PATH = WORKING_DIR / "CVES_Full_Workspace_For_Git.zip"
LIGHT_ZIP_PATH = WORKING_DIR / "CVES_Light_For_Git.zip"

# Recommended .gitignore content bundled into the archive for safe Git uploads
GITIGNORE_CONTENT = """# Raw PhysioNet dataset & large archives
cves-dataset/
*.zip
*.dat
*.hea

# Python & Jupyter artifacts
__pycache__/
*.py[cod]
.ipynb_checkpoints/
.virtual_documents/

# OS junk
.DS_Store
Thumbs.db
"""


# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================
def get_dir_stats(directory: Path) -> tuple[int, float]:
    """Returns (file_count, total_size_mb) for a given directory."""
    if not directory.exists():
        return 0, 0.0
    file_count = 0
    total_bytes = 0
    for root, _, filenames in os.walk(directory):
        for fname in filenames:
            fpath = Path(root) / fname
            if fpath.is_file():
                file_count += 1
                total_bytes += fpath.stat().st_size
    return file_count, total_bytes / (1024 * 1024)


def build_clean_zip(
    source_dir: Path,
    output_zip: Path,
    exclude_subdirs: set[str] | None = None,
    include_gitignore: bool = True,
) -> tuple[int, float]:
    """
    Zips source_dir using clean relative paths (no /kaggle/working prefix).
    Returns (files_added, archive_size_mb).
    """
    exclude_subdirs = exclude_subdirs or set()
    files_added = 0

    with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, filenames in os.walk(source_dir):
            # Prune excluded subdirectories and cache folders in-place
            dirs[:] = [
                d for d in dirs
                if d not in exclude_subdirs
                and d not in {"__pycache__", ".ipynb_checkpoints"}
            ]
            for fname in sorted(filenames):
                if fname.startswith("."):
                    continue
                fpath = Path(root) / fname
                # Preserve clean relative structure starting inside workspace
                arcname = fpath.relative_to(source_dir)
                zipf.write(fpath, arcname=str(arcname))
                files_added += 1

        # Add a ready-to-use .gitignore at the root of the archive
        if include_gitignore:
            zipf.writestr(".gitignore", GITIGNORE_CONTENT)

    size_mb = output_zip.stat().st_size / (1024 * 1024)
    return files_added, size_mb


# =============================================================================
# 3. MAIN PACKAGING WORKFLOW
# =============================================================================
def main() -> None:
    if not WORKSPACE_DIR.exists():
        raise FileNotFoundError(
            f"Workspace folder not found at {WORKSPACE_DIR}. "
            "Verify that the pipeline completed and check /kaggle/working."
        )

    print("=" * 68)
    print(f"Inspecting Workspace: {WORKSPACE_DIR}")
    print("=" * 68)

    for sub in EXPECTED_SUBDIRS:
        sub_path = WORKSPACE_DIR / sub
        n_files, mb = get_dir_stats(sub_path)
        status = f"{n_files:>4} files | {mb:>7.2f} MB" if sub_path.exists() else "MISSING"
        print(f"  ├── {sub:<28} [{status}]")

    for fname in EXPECTED_FILES:
        fpath = WORKSPACE_DIR / fname
        if fpath.exists():
            kb = fpath.stat().st_size / 1024
            print(f"  └── {fname:<28} [   1 file  | {kb:>7.2f} KB]")
        else:
            print(f"  └── {fname:<28} [MISSING]")

    print("-" * 68)

    # 1. Build Full Workspace ZIP (includes all 5 folders + run_log.txt)
    try:
        full_count, full_mb = build_clean_zip(
            source_dir=WORKSPACE_DIR,
            output_zip=FULL_ZIP_PATH,
            exclude_subdirs=set(),
        )
        print(f"[OK] Full Archive  : {FULL_ZIP_PATH.name} ({full_count} files, {full_mb:.2f} MB)")
        if full_mb > 900:
            print("     [NOTE] Full archive is >900 MB. GitHub recommends keeping repos <1 GB.")
            print("            Consider uploading the Light archive to GitHub instead.")
    except Exception as exc:
        print(f"[ERROR] Failed to build full archive: {exc}")

    # 2. Build Lightweight Git-Safe ZIP (excludes Supplementary_QC_Plots)
    try:
        light_count, light_mb = build_clean_zip(
            source_dir=WORKSPACE_DIR,
            output_zip=LIGHT_ZIP_PATH,
            exclude_subdirs={"Supplementary_QC_Plots"},
        )
        print(f"[OK] Light Archive : {LIGHT_ZIP_PATH.name} ({light_count} files, {light_mb:.2f} MB)")
    except Exception as exc:
        print(f"[ERROR] Failed to build light archive: {exc}")

    print("=" * 68)

    # 3. Display clickable download links inside Kaggle Notebook
    try:
        from IPython.display import FileLink, display

        print("Click a link below to download to your local machine (or use the Kaggle Output sidebar):")
        if FULL_ZIP_PATH.exists():
            display(FileLink(FULL_ZIP_PATH.name))
        if LIGHT_ZIP_PATH.exists():
            display(FileLink(LIGHT_ZIP_PATH.name))
    except ImportError:
        print("Not running in an interactive notebook. Download the ZIP files directly from /kaggle/working/.")


if __name__ == "__main__":
    main()

Inspecting Workspace: /kaggle/working/cves_nabian2018_ccs_workspace
  ├── Figures_Publication          [   4 files |    1.36 MB]
  ├── Figures_Supplementary        [   6 files |    2.39 MB]
  ├── Supplementary_QC_Plots       [ 784 files |  619.27 MB]
  ├── Supplementary_Tachograms     [ 392 files |    0.73 MB]
  ├── Tables_CSV                   [   9 files |    0.72 MB]
  └── run_log.txt                  [   1 file  |    6.18 KB]
--------------------------------------------------------------------
[OK] Full Archive  : CVES_Full_Workspace_For_Git.zip (1196 files, 567.66 MB)
[OK] Light Archive : CVES_Light_For_Git.zip (412 files, 3.93 MB)
Click a link below to download to your local machine (or use the Kaggle Output sidebar):


/kaggle/working/CVES_Full_Workspace_For_Git.zip

/kaggle/working/CVES_Light_For_Git.zip

In [4]:
!zip -r -q /kaggle/working/all_cves_outputs.zip /kaggle/working/cves_nabian2018_ccs_workspace

from IPython.display import FileLink
FileLink("all_cves_outputs.zip")

/kaggle/working/all_cves_outputs.zip

In [4]:
from pathlib import Path

qc_dir = Path("/kaggle/working/cves_nabian2018_ccs_workspace/Supplementary_QC_Plots")
pngs = list(qc_dir.rglob("*.png"))
print(f"QC PNGs saved so far: {len(pngs)} (expected ~784 for 392 epochs)")

log_file = Path("/kaggle/working/cves_nabian2018_ccs_workspace/run_log.txt")
if log_file.exists():
    print("\n--- Last 10 lines of run_log.txt ---")
    print("\n".join(log_file.read_text().splitlines()[-10:]))

QC PNGs saved so far: 784 (expected ~784 for 392 epochs)

--- Last 10 lines of run_log.txt ---
17:28:21  [INFO]  Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
17:28:21  [INFO]  Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
17:28:21  [INFO]  Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
17:28:21  [INFO]  Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
17:28:21  [INFO]  Using categorical units to plot a list of strings that 

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Automated GitHub Repository Scaffold Builder
Organizes downloaded Kaggle/Colab outputs and generates all required GitHub
root & documentation files:
    - requirements.txt
    - .gitignore
    - LICENSE (MIT)
    - CITATION.cff
    - docs/DATA_DICTIONARY.md
    - Full folder hierarchy (src/, docs/, results/tables/, results/figures_publication/,
      results/figures_supplementary/, results/supplementary_tachograms/, logs/)

Usage:
    python build_github_repo.py --source ./unzipped_kaggle_outputs --target ./cves-stroke-autonomic-complexity
"""

import argparse
import shutil
import sys
from pathlib import Path

# =============================================================================
# 1. STANDARD GITHUB COMPANION FILE TEMPLATES
# =============================================================================
REQUIREMENTS_TXT = """# Core scientific computing
numpy>=1.24.0
pandas>=2.0.0
scipy>=1.10.0

# Signal processing & PhysioNet WFDB I/O
neurokit2>=0.2.7
wfdb>=4.1.0
awscli>=1.29.0

# Hierarchical mixed-effects modeling & diagnostics
statsmodels>=0.14.0
patsy>=0.5.3

# Publication visualization & progress reporting
matplotlib>=3.7.0
seaborn>=0.13.0
tqdm>=4.65.0
"""

GITIGNORE_TXT = """# Raw PhysioNet CVES binary waveforms & clinical registry (download via AWS S3)
cves-dataset/
*.dat
*.hea
subjects.csv

# Large archives & optional bulk QC plots (~784 PNGs; archive on OSF/Zenodo instead)
*.zip
*.tar.gz
Supplementary_QC_Plots/
results/supplementary_qc_plots/

# Python environments & caches
.venv/
venv/
__pycache__/
*.py[cod]
*$py.class
.pytest_cache/

# Jupyter, Kaggle & IDE metadata
.ipynb_checkpoints/
.virtual_documents/
.vscode/
.idea/

# OS metadata
.DS_Store
Thumbs.db
"""

MIT_LICENSE_TXT = """MIT License

Copyright (c) 2026 Aswin Tony Joseph

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""

CITATION_CFF_TXT = """cff-version: 1.2.0
message: "If you use this pipeline or derived feature tables, please cite this software and the PhysioNet CVES dataset."
title: "Hemispheric Dissociation of Multiscale Autonomic Complexity and Cardiovagal Reserve in Chronic Ischemic Stroke (CCS 1000 Hz Pipeline)"
version: "1.0.0"
date-released: "2026-11-20"
license: MIT
authors:
  - family-names: "Joseph"
    given-names: "Aswin Tony"
    email: "aswintony7@gmail.com"
    affiliation: "Government Medical College Thiruvananthapuram, Kerala, India"
  - family-names: "Venugopal"
    given-names: "Rahul"
    affiliation: "Centre for Consciousness Studies, Department of Neurophysiology, NIMHANS, Bengaluru, India"
keywords:
  - chronic ischemic stroke
  - heart rate variability
  - multiscale entropy
  - deceleration capacity
  - linear mixed-effects models
  - PhysioNet CVES
"""

DATA_DICTIONARY_MD = """# Data Dictionary — Exported Result Tables (`results/tables/`)

## 1. Signal Quality & Epoch Audit Tables
- **`Table_S0_Epoch_Sanity_Audit.csv`**: Automated comparison of extracted `HR_bpm` against `subjects.csv` reference HR columns (`Pearson_r`, `Spearman_rs`, `MAE_bpm`, `Bias_bpm`, `Med_RMSSD_ms`, `Med_DFA_a1`, `Pass_Check`).
- **`Table_S0B_HR_Outliers_gt5bpm.csv`**: Subject-level list of any epoch differing by >5 bpm from `subjects.csv` (includes `Extracted_HR`, `GT_HR`, `Diff_bpm`, `RMSSD_ms`, `SQI`).
- **`Table_S1_Master_Epoch_Features_Nabian2018_1000Hz.csv`**: Full 392-epoch extracted feature table across 75 subjects (114 columns: metadata, SQI, RR quality flags, HR, EDR respiration, PRSA DC/AC, Coarse MSE 6–15, and full NeuroKit2 linear/spectral/nonlinear HRV battery).

## 2. Multicollinearity & LMEM Diagnostic Tables
- **`Table_S2_VIF_Multicollinearity.csv`**: Variance Inflation Factors (`VIF`) and `Tolerance` (`1 / VIF`) across the main-effects LMEM design matrix (all predictors pass `VIF < 5.0`, range `1.003–1.759`).
- **`Table_S3_LMEM_Diagnostics_ICC_R2_Shapiro.csv`**: Random-intercept variance (`Var_u0`), residual variance (`Var_eps`), Intraclass Correlation Coefficient (`ICC`), Nakagawa–Schielzeth Marginal and Conditional $R^2$, REML log-likelihood, and Shapiro–Wilk residual normality statistics across both modeling suites.

## 3. Primary Manuscript Statistical Tables
- **`Table1_Demographics_and_Resting_Subtypes.csv`**: Cohort demographics ($N = 72$) and resting baseline (`Stress_Score == 0`) subtype means and SDs.
- **`Table1B_Resting_Hypothesis_Tests.csv`**: Resting 4-group ANOVAs and Welch/Mann–Whitney pairwise comparisons for H1 (Multiscale Complexity Ratios) and H2 (`PRSA_DC_clean`).
- **`Table2_Task_State_Descriptives.csv`**: Descriptive statistics (`count`, `mean`, `std`, `median`) across all six protocol tasks (States 0–3).
- **`Table3_LMEM_StrictSinus_Orthostatic.csv`**: REML LMEM parameter estimates (`Beta`, `SE`, `z_stat`, `p_value`, 95% CI, `ICC`) across the strict-sinus orthostatic transition ($N = 236$ epochs, $67$ subjects).
- **`Table4_LMEM_4State_Continuum.csv`**: REML LMEM parameter estimates across the calibrated 4-state stress continuum ($N = 377\\text{–}386$ epochs, $72$ subjects).
- **`Table5_Translational_Spearman_Coupling.csv`**: Two-tailed Spearman rank correlations ($r_s$, $R^2$ rank %, $p$-value, $N$) linking resting and Empirical Bayes reactivity slopes (`u1i_*`) to TCD CBFV, MMPF phase shift, Trail Making Part B, 12-min hallway gait velocity, and NIHSS.
"""

# Mapping from Kaggle workspace folder names to clean GitHub repo paths
FOLDER_MAP = {
    "Tables_CSV": "results/tables",
    "Figures_Publication": "results/figures_publication",
    "Figures_Supplementary": "results/figures_supplementary",
    "Supplementary_Tachograms": "results/supplementary_tachograms",
}


# =============================================================================
# 2. REPOSITORY BUILDER LOGIC
# =============================================================================
def create_directory_tree(repo_root: Path) -> None:
    """Creates the standardized GitHub folder hierarchy."""
    subdirs = [
        "src",
        "docs",
        "results/tables",
        "results/figures_publication",
        "results/figures_supplementary",
        "results/supplementary_tachograms",
        "logs",
    ]
    for rel in subdirs:
        (repo_root / rel).mkdir(parents=True, exist_ok=True)


def write_companion_files(repo_root: Path) -> None:
    """Writes requirements.txt, .gitignore, LICENSE, CITATION.cff, and DATA_DICTIONARY.md."""
    files_to_write = {
        "requirements.txt": REQUIREMENTS_TXT,
        ".gitignore": GITIGNORE_TXT,
        "LICENSE": MIT_LICENSE_TXT,
        "CITATION.cff": CITATION_CFF_TXT,
        "docs/DATA_DICTIONARY.md": DATA_DICTIONARY_MD,
    }
    for rel_path, content in files_to_write.items():
        dest = repo_root / rel_path
        dest.write_text(content.strip() + "\n", encoding="utf-8")
        print(f"  [WRITE] {rel_path}")


def copy_kaggle_outputs(source_dir: Path, repo_root: Path) -> None:
    """Copies unzipped Kaggle workspace folders and logs into their clean GitHub paths."""
    if not source_dir.exists():
        print(f"  [WARN] Source directory not found: {source_dir} (created empty scaffold only).")
        return

    # Copy mapped output folders
    for src_sub, dst_rel in FOLDER_MAP.items():
        src_path = source_dir / src_sub
        dst_path = repo_root / dst_rel
        if src_path.exists() and src_path.is_dir():
            count = 0
            for item in sorted(src_path.iterdir()):
                if item.is_file() and not item.name.startswith("."):
                    shutil.copy2(item, dst_path / item.name)
                    count += 1
            print(f"  [COPY]  {src_sub:<26} -> {dst_rel} ({count} files)")

    # Copy run_log.txt if present
    log_src = source_dir / "run_log.txt"
    if log_src.exists():
        shutil.copy2(log_src, repo_root / "logs" / "run_log.txt")
        print("  [COPY]  run_log.txt                -> logs/run_log.txt")

    # Copy pipeline Python scripts if present in source_dir
    for py_file in sorted(source_dir.glob("*.py")):
        if py_file.name != Path(__file__).name:
            shutil.copy2(py_file, repo_root / "src" / py_file.name)
            print(f"  [COPY]  {py_file.name:<26} -> src/{py_file.name}")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Build a clean GitHub repository scaffold for the CVES Stroke HRV project."
    )
    parser.add_argument(
        "--source",
        type=Path,
        default=Path("./cves_nabian2018_ccs_workspace"),
        help="Path to the unzipped Kaggle/Colab workspace folder.",
    )
    parser.add_argument(
        "--target",
        type=Path,
        default=Path("./cves-stroke-autonomic-complexity"),
        help="Target path for the new GitHub repository folder.",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    repo_root = args.target.resolve()
    source_dir = args.source.resolve()

    print("=" * 72)
    print(f"Building GitHub Repository Scaffold at: {repo_root}")
    print("=" * 72)

    try:
        create_directory_tree(repo_root)
        write_companion_files(repo_root)
        copy_kaggle_outputs(source_dir, repo_root)
    except Exception as exc:
        print(f"[ERROR] Failed to construct repository scaffold: {exc}", file=sys.stderr)
        sys.exit(1)

    print("-" * 72)
    print("[DONE] Repository scaffold ready. Next Git commands:")
    print(f"       cd {repo_root}")
    print("       # Place README.md at root, LMEM_Justification_and_Code_Reasoning.md in docs/,")
    print("       # and cves_stroke_hrv_nabian2018_ccs_pipeline_v2.py in src/.")
    print("       git init && git branch -M main")
    print("       git add . && git commit -m 'Initial release: 1000 Hz CCS Nabian2018 stroke HRV pipeline'")
    print("       git remote add origin https://github.com/<your-username>/cves-stroke-autonomic-complexity.git")
    print("       git push -u origin main")
    print("=" * 72)


if __name__ == "__main__":
    main()

usage: colab_kernel_launcher.py [-h] [--source SOURCE] [--target TARGET]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-b97bebe7-324b-4924-962e-bf14ec944a8e.json


SystemExit: 2

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Automated GitHub Repository Scaffold Builder
Organizes downloaded Kaggle/Colab outputs and generates all required GitHub
root & documentation files:
    - requirements.txt
    - .gitignore
    - LICENSE (MIT)
    - CITATION.cff
    - docs/DATA_DICTIONARY.md
    - Full folder hierarchy (src/, docs/, results/tables/, results/figures_publication/,
      results/figures_supplementary/, results/supplementary_tachograms/, logs/)

Usage:
    python build_github_repo.py --source ./unzipped_kaggle_outputs --target ./cves-stroke-autonomic-complexity
"""

import argparse
import shutil
import sys
from pathlib import Path

# =============================================================================
# 1. STANDARD GITHUB COMPANION FILE TEMPLATES
# =============================================================================
REQUIREMENTS_TXT = """# Core scientific computing
numpy>=1.24.0
pandas>=2.0.0
scipy>=1.10.0

# Signal processing & PhysioNet WFDB I/O
neurokit2>=0.2.7
wfdb>=4.1.0
awscli>=1.29.0

# Hierarchical mixed-effects modeling & diagnostics
statsmodels>=0.14.0
patsy>=0.5.3

# Publication visualization & progress reporting
matplotlib>=3.7.0
seaborn>=0.13.0
tqdm>=4.65.0
"""

GITIGNORE_TXT = """# Raw PhysioNet CVES binary waveforms & clinical registry (download via AWS S3)
cves-dataset/
*.dat
*.hea
subjects.csv

# Large archives & optional bulk QC plots (~784 PNGs; archive on OSF/Zenodo instead)
*.zip
*.tar.gz
Supplementary_QC_Plots/
results/supplementary_qc_plots/

# Python environments & caches
.venv/
venv/
__pycache__/
*.py[cod]
*$py.class
.pytest_cache/

# Jupyter, Kaggle & IDE metadata
.ipynb_checkpoints/
.virtual_documents/
.vscode/
.idea/

# OS metadata
.DS_Store
Thumbs.db
"""

MIT_LICENSE_TXT = """MIT License

Copyright (c) 2026 Aswin Tony Joseph

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""

CITATION_CFF_TXT = """cff-version: 1.2.0
message: "If you use this pipeline or derived feature tables, please cite this software and the PhysioNet CVES dataset."
title: "Hemispheric Dissociation of Multiscale Autonomic Complexity and Cardiovagal Reserve in Chronic Ischemic Stroke (CCS 1000 Hz Pipeline)"
version: "1.0.0"
date-released: "2026-11-20"
license: MIT
authors:
  - family-names: "Joseph"
    given-names: "Aswin Tony"
    email: "aswintony7@gmail.com"
    affiliation: "Government Medical College Thiruvananthapuram, Kerala, India"
  - family-names: "Venugopal"
    given-names: "Rahul"
    affiliation: "Centre for Consciousness Studies, Department of Neurophysiology, NIMHANS, Bengaluru, India"
keywords:
  - chronic ischemic stroke
  - heart rate variability
  - multiscale entropy
  - deceleration capacity
  - linear mixed-effects models
  - PhysioNet CVES
"""

DATA_DICTIONARY_MD = """# Data Dictionary — Exported Result Tables (`results/tables/`)

## 1. Signal Quality & Epoch Audit Tables
- **`Table_S0_Epoch_Sanity_Audit.csv`**: Automated comparison of extracted `HR_bpm` against `subjects.csv` reference HR columns (`Pearson_r`, `Spearman_rs`, `MAE_bpm`, `Bias_bpm`, `Med_RMSSD_ms`, `Med_DFA_a1`, `Pass_Check`).
- **`Table_S0B_HR_Outliers_gt5bpm.csv`**: Subject-level list of any epoch differing by >5 bpm from `subjects.csv` (includes `Extracted_HR`, `GT_HR`, `Diff_bpm`, `RMSSD_ms`, `SQI`).
- **`Table_S1_Master_Epoch_Features_Nabian2018_1000Hz.csv`**: Full 392-epoch extracted feature table across 75 subjects (114 columns: metadata, SQI, RR quality flags, HR, EDR respiration, PRSA DC/AC, Coarse MSE 6–15, and full NeuroKit2 linear/spectral/nonlinear HRV battery).

## 2. Multicollinearity & LMEM Diagnostic Tables
- **`Table_S2_VIF_Multicollinearity.csv`**: Variance Inflation Factors (`VIF`) and `Tolerance` (`1 / VIF`) across the main-effects LMEM design matrix (all predictors pass `VIF < 5.0`, range `1.003–1.759`).
- **`Table_S3_LMEM_Diagnostics_ICC_R2_Shapiro.csv`**: Random-intercept variance (`Var_u0`), residual variance (`Var_eps`), Intraclass Correlation Coefficient (`ICC`), Nakagawa–Schielzeth Marginal and Conditional $R^2$, REML log-likelihood, and Shapiro–Wilk residual normality statistics across both modeling suites.

## 3. Primary Manuscript Statistical Tables
- **`Table1_Demographics_and_Resting_Subtypes.csv`**: Cohort demographics ($N = 72$) and resting baseline (`Stress_Score == 0`) subtype means and SDs.
- **`Table1B_Resting_Hypothesis_Tests.csv`**: Resting 4-group ANOVAs and Welch/Mann–Whitney pairwise comparisons for H1 (Multiscale Complexity Ratios) and H2 (`PRSA_DC_clean`).
- **`Table2_Task_State_Descriptives.csv`**: Descriptive statistics (`count`, `mean`, `std`, `median`) across all six protocol tasks (States 0–3).
- **`Table3_LMEM_StrictSinus_Orthostatic.csv`**: REML LMEM parameter estimates (`Beta`, `SE`, `z_stat`, `p_value`, 95% CI, `ICC`) across the strict-sinus orthostatic transition ($N = 236$ epochs, $67$ subjects).
- **`Table4_LMEM_4State_Continuum.csv`**: REML LMEM parameter estimates across the calibrated 4-state stress continuum ($N = 377\\text{–}386$ epochs, $72$ subjects).
- **`Table5_Translational_Spearman_Coupling.csv`**: Two-tailed Spearman rank correlations ($r_s$, $R^2$ rank %, $p$-value, $N$) linking resting and Empirical Bayes reactivity slopes (`u1i_*`) to TCD CBFV, MMPF phase shift, Trail Making Part B, 12-min hallway gait velocity, and NIHSS.
"""

# Mapping from Kaggle workspace folder names to clean GitHub repo paths
FOLDER_MAP = {
    "Tables_CSV": "results/tables",
    "Figures_Publication": "results/figures_publication",
    "Figures_Supplementary": "results/figures_supplementary",
    "Supplementary_Tachograms": "results/supplementary_tachograms",
}


# =============================================================================
# 2. REPOSITORY BUILDER LOGIC
# =============================================================================
def create_directory_tree(repo_root: Path) -> None:
    """Creates the standardized GitHub folder hierarchy."""
    subdirs = [
        "src",
        "docs",
        "results/tables",
        "results/figures_publication",
        "results/figures_supplementary",
        "results/supplementary_tachograms",
        "logs",
    ]
    for rel in subdirs:
        (repo_root / rel).mkdir(parents=True, exist_ok=True)


def write_companion_files(repo_root: Path) -> None:
    """Writes requirements.txt, .gitignore, LICENSE, CITATION.cff, and DATA_DICTIONARY.md."""
    files_to_write = {
        "requirements.txt": REQUIREMENTS_TXT,
        ".gitignore": GITIGNORE_TXT,
        "LICENSE": MIT_LICENSE_TXT,
        "CITATION.cff": CITATION_CFF_TXT,
        "docs/DATA_DICTIONARY.md": DATA_DICTIONARY_MD,
    }
    for rel_path, content in files_to_write.items():
        dest = repo_root / rel_path
        dest.write_text(content.strip() + "\n", encoding="utf-8")
        print(f"  [WRITE] {rel_path}")


def copy_kaggle_outputs(source_dir: Path, repo_root: Path) -> None:
    """Copies unzipped Kaggle workspace folders and logs into their clean GitHub paths."""
    if not source_dir.exists():
        print(f"  [WARN] Source directory not found: {source_dir} (created empty scaffold only).")
        return

    # Copy mapped output folders
    for src_sub, dst_rel in FOLDER_MAP.items():
        src_path = source_dir / src_sub
        dst_path = repo_root / dst_rel
        if src_path.exists() and src_path.is_dir():
            count = 0
            for item in sorted(src_path.iterdir()):
                if item.is_file() and not item.name.startswith("."):
                    shutil.copy2(item, dst_path / item.name)
                    count += 1
            print(f"  [COPY]  {src_sub:<26} -> {dst_rel} ({count} files)")

    # Copy run_log.txt if present
    log_src = source_dir / "run_log.txt"
    if log_src.exists():
        shutil.copy2(log_src, repo_root / "logs" / "run_log.txt")
        print("  [COPY]  run_log.txt                -> logs/run_log.txt")

    # Copy pipeline Python scripts if present in source_dir
    for py_file in sorted(source_dir.glob("*.py")):
        if py_file.name != Path(__file__).name:
            shutil.copy2(py_file, repo_root / "src" / py_file.name)
            print(f"  [COPY]  {py_file.name:<26} -> src/{py_file.name}")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Build a clean GitHub repository scaffold for the CVES Stroke HRV project."
    )
    parser.add_argument(
        "--source",
        type=Path,
        default=Path("./cves_nabian2018_ccs_workspace"),
        help="Path to the unzipped Kaggle/Colab workspace folder.",
    )
    parser.add_argument(
        "--target",
        type=Path,
        default=Path("./cves-stroke-autonomic-complexity"),
        help="Target path for the new GitHub repository folder.",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    repo_root = args.target.resolve()
    source_dir = args.source.resolve()

    print("=" * 72)
    print(f"Building GitHub Repository Scaffold at: {repo_root}")
    print("=" * 72)

    try:
        create_directory_tree(repo_root)
        write_companion_files(repo_root)
        copy_kaggle_outputs(source_dir, repo_root)
    except Exception as exc:
        print(f"[ERROR] Failed to construct repository scaffold: {exc}", file=sys.stderr)
        sys.exit(1)

    print("-" * 72)
    print("[DONE] Repository scaffold ready. Next Git commands:")
    print(f"       cd {repo_root}")
    print("       # Place README.md at root, LMEM_Justification_and_Code_Reasoning.md in docs/,")
    print("       # and cves_stroke_hrv_nabian2018_ccs_pipeline_v2.py in src/.")
    print("       git init && git branch -M main")
    print("       git add . && git commit -m 'Initial release: 1000 Hz CCS Nabian2018 stroke HRV pipeline'")
    print("       git remote add origin https://github.com/<your-username>/cves-stroke-autonomic-complexity.git")
    print("       git push -u origin main")
    print("=" * 72)


if __name__ == "__main__":
    main()

usage: colab_kernel_launcher.py [-h] [--source SOURCE] [--target TARGET]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-57f07cc1-188e-42fc-b36a-82623e700639.json


SystemExit: 2

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Automated GitHub Repository Scaffold Builder
Organizes downloaded Kaggle/Colab outputs and generates all required GitHub
root & documentation files:
    - requirements.txt
    - .gitignore
    - LICENSE (MIT)
    - CITATION.cff
    - docs/DATA_DICTIONARY.md
    - Full folder hierarchy (src/, docs/, results/tables/, results/figures_publication/,
      results/figures_supplementary/, results/supplementary_tachograms/, logs/)

Usage:
    python build_github_repo.py --source ./unzipped_kaggle_outputs --target ./cves-stroke-autonomic-complexity
"""

import argparse
import shutil
import sys
from pathlib import Path

# =============================================================================
# 1. STANDARD GITHUB COMPANION FILE TEMPLATES
# =============================================================================
REQUIREMENTS_TXT = """# Core scientific computing
numpy>=1.24.0
pandas>=2.0.0
scipy>=1.10.0

# Signal processing & PhysioNet WFDB I/O
neurokit2>=0.2.7
wfdb>=4.1.0
awscli>=1.29.0

# Hierarchical mixed-effects modeling & diagnostics
statsmodels>=0.14.0
patsy>=0.5.3

# Publication visualization & progress reporting
matplotlib>=3.7.0
seaborn>=0.13.0
tqdm>=4.65.0
"""

GITIGNORE_TXT = """# Raw PhysioNet CVES binary waveforms & clinical registry (download via AWS S3)
cves-dataset/
*.dat
*.hea
subjects.csv

# Large archives & optional bulk QC plots (~784 PNGs; archive on OSF/Zenodo instead)
*.zip
*.tar.gz
Supplementary_QC_Plots/
results/supplementary_qc_plots/

# Python environments & caches
.venv/
venv/
__pycache__/
*.py[cod]
*$py.class
.pytest_cache/

# Jupyter, Kaggle & IDE metadata
.ipynb_checkpoints/
.virtual_documents/
.vscode/
.idea/

# OS metadata
.DS_Store
Thumbs.db
"""

MIT_LICENSE_TXT = """MIT License

Copyright (c) 2026 Aswin Tony Joseph

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""

CITATION_CFF_TXT = """cff-version: 1.2.0
message: "If you use this pipeline or derived feature tables, please cite this software and the PhysioNet CVES dataset."
title: "Hemispheric Dissociation of Multiscale Autonomic Complexity and Cardiovagal Reserve in Chronic Ischemic Stroke (CCS 1000 Hz Pipeline)"
version: "1.0.0"
date-released: "2026-11-20"
license: MIT
authors:
  - family-names: "Joseph"
    given-names: "Aswin Tony"
    email: "aswintony7@gmail.com"
    affiliation: "Government Medical College Thiruvananthapuram, Kerala, India"
  - family-names: "Venugopal"
    given-names: "Rahul"
    affiliation: "Centre for Consciousness Studies, Department of Neurophysiology, NIMHANS, Bengaluru, India"
keywords:
  - chronic ischemic stroke
  - heart rate variability
  - multiscale entropy
  - deceleration capacity
  - linear mixed-effects models
  - PhysioNet CVES
"""

DATA_DICTIONARY_MD = """# Data Dictionary — Exported Result Tables (`results/tables/`)

## 1. Signal Quality & Epoch Audit Tables
- **`Table_S0_Epoch_Sanity_Audit.csv`**: Automated comparison of extracted `HR_bpm` against `subjects.csv` reference HR columns (`Pearson_r`, `Spearman_rs`, `MAE_bpm`, `Bias_bpm`, `Med_RMSSD_ms`, `Med_DFA_a1`, `Pass_Check`).
- **`Table_S0B_HR_Outliers_gt5bpm.csv`**: Subject-level list of any epoch differing by >5 bpm from `subjects.csv` (includes `Extracted_HR`, `GT_HR`, `Diff_bpm`, `RMSSD_ms`, `SQI`).
- **`Table_S1_Master_Epoch_Features_Nabian2018_1000Hz.csv`**: Full 392-epoch extracted feature table across 75 subjects (114 columns: metadata, SQI, RR quality flags, HR, EDR respiration, PRSA DC/AC, Coarse MSE 6–15, and full NeuroKit2 linear/spectral/nonlinear HRV battery).

## 2. Multicollinearity & LMEM Diagnostic Tables
- **`Table_S2_VIF_Multicollinearity.csv`**: Variance Inflation Factors (`VIF`) and `Tolerance` (`1 / VIF`) across the main-effects LMEM design matrix (all predictors pass `VIF < 5.0`, range `1.003–1.759`).
- **`Table_S3_LMEM_Diagnostics_ICC_R2_Shapiro.csv`**: Random-intercept variance (`Var_u0`), residual variance (`Var_eps`), Intraclass Correlation Coefficient (`ICC`), Nakagawa–Schielzeth Marginal and Conditional $R^2$, REML log-likelihood, and Shapiro–Wilk residual normality statistics across both modeling suites.

## 3. Primary Manuscript Statistical Tables
- **`Table1_Demographics_and_Resting_Subtypes.csv`**: Cohort demographics ($N = 72$) and resting baseline (`Stress_Score == 0`) subtype means and SDs.
- **`Table1B_Resting_Hypothesis_Tests.csv`**: Resting 4-group ANOVAs and Welch/Mann–Whitney pairwise comparisons for H1 (Multiscale Complexity Ratios) and H2 (`PRSA_DC_clean`).
- **`Table2_Task_State_Descriptives.csv`**: Descriptive statistics (`count`, `mean`, `std`, `median`) across all six protocol tasks (States 0–3).
- **`Table3_LMEM_StrictSinus_Orthostatic.csv`**: REML LMEM parameter estimates (`Beta`, `SE`, `z_stat`, `p_value`, 95% CI, `ICC`) across the strict-sinus orthostatic transition ($N = 236$ epochs, $67$ subjects).
- **`Table4_LMEM_4State_Continuum.csv`**: REML LMEM parameter estimates across the calibrated 4-state stress continuum ($N = 377\\text{–}386$ epochs, $72$ subjects).
- **`Table5_Translational_Spearman_Coupling.csv`**: Two-tailed Spearman rank correlations ($r_s$, $R^2$ rank %, $p$-value, $N$) linking resting and Empirical Bayes reactivity slopes (`u1i_*`) to TCD CBFV, MMPF phase shift, Trail Making Part B, 12-min hallway gait velocity, and NIHSS.
"""

# Mapping from Kaggle workspace folder names to clean GitHub repo paths
FOLDER_MAP = {
    "Tables_CSV": "results/tables",
    "Figures_Publication": "results/figures_publication",
    "Figures_Supplementary": "results/figures_supplementary",
    "Supplementary_Tachograms": "results/supplementary_tachograms",
}


# =============================================================================
# 2. REPOSITORY BUILDER LOGIC
# =============================================================================
def create_directory_tree(repo_root: Path) -> None:
    """Creates the standardized GitHub folder hierarchy."""
    subdirs = [
        "src",
        "docs",
        "results/tables",
        "results/figures_publication",
        "results/figures_supplementary",
        "results/supplementary_tachograms",
        "logs",
    ]
    for rel in subdirs:
        (repo_root / rel).mkdir(parents=True, exist_ok=True)


def write_companion_files(repo_root: Path) -> None:
    """Writes requirements.txt, .gitignore, LICENSE, CITATION.cff, and DATA_DICTIONARY.md."""
    files_to_write = {
        "requirements.txt": REQUIREMENTS_TXT,
        ".gitignore": GITIGNORE_TXT,
        "LICENSE": MIT_LICENSE_TXT,
        "CITATION.cff": CITATION_CFF_TXT,
        "docs/DATA_DICTIONARY.md": DATA_DICTIONARY_MD,
    }
    for rel_path, content in files_to_write.items():
        dest = repo_root / rel_path
        dest.write_text(content.strip() + "\n", encoding="utf-8")
        print(f"  [WRITE] {rel_path}")


def copy_kaggle_outputs(source_dir: Path, repo_root: Path) -> None:
    """Copies unzipped Kaggle workspace folders and logs into their clean GitHub paths."""
    if not source_dir.exists():
        print(f"  [WARN] Source directory not found: {source_dir} (created empty scaffold only).")
        return

    # Copy mapped output folders
    for src_sub, dst_rel in FOLDER_MAP.items():
        src_path = source_dir / src_sub
        dst_path = repo_root / dst_rel
        if src_path.exists() and src_path.is_dir():
            count = 0
            for item in sorted(src_path.iterdir()):
                if item.is_file() and not item.name.startswith("."):
                    shutil.copy2(item, dst_path / item.name)
                    count += 1
            print(f"  [COPY]  {src_sub:<26} -> {dst_rel} ({count} files)")

    # Copy run_log.txt if present
    log_src = source_dir / "run_log.txt"
    if log_src.exists():
        shutil.copy2(log_src, repo_root / "logs" / "run_log.txt")
        print("  [COPY]  run_log.txt                -> logs/run_log.txt")

    # Copy pipeline Python scripts if present in source_dir
    for py_file in sorted(source_dir.glob("*.py")):
        if py_file.name != Path(__file__).name:
            shutil.copy2(py_file, repo_root / "src" / py_file.name)
            print(f"  [COPY]  {py_file.name:<26} -> src/{py_file.name}")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Build a clean GitHub repository scaffold for the CVES Stroke HRV project."
    )
    parser.add_argument(
        "--source",
        type=Path,
        default=Path("./cves_nabian2018_ccs_workspace"),
        help="Path to the unzipped Kaggle/Colab workspace folder.",
    )
    parser.add_argument(
        "--target",
        type=Path,
        default=Path("./cves-stroke-autonomic-complexity"),
        help="Target path for the new GitHub repository folder.",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    repo_root = args.target.resolve()
    source_dir = args.source.resolve()

    print("=" * 72)
    print(f"Building GitHub Repository Scaffold at: {repo_root}")
    print("=" * 72)

    try:
        create_directory_tree(repo_root)
        write_companion_files(repo_root)
        copy_kaggle_outputs(source_dir, repo_root)
    except Exception as exc:
        print(f"[ERROR] Failed to construct repository scaffold: {exc}", file=sys.stderr)
        sys.exit(1)

    print("-" * 72)
    print("[DONE] Repository scaffold ready. Next Git commands:")
    print(f"       cd {repo_root}")
    print("       # Place README.md at root, LMEM_Justification_and_Code_Reasoning.md in docs/,")
    print("       # and cves_stroke_hrv_nabian2018_ccs_pipeline_v2.py in src/.")
    print("       git init && git branch -M main")
    print("       git add . && git commit -m 'Initial release: 1000 Hz CCS Nabian2018 stroke HRV pipeline'")
    print("       git remote add origin https://github.com/<your-username>/cves-stroke-autonomic-complexity.git")
    print("       git push -u origin main")
    print("=" * 72)


if __name__ == "__main__":
    main()

usage: colab_kernel_launcher.py [-h] [--source SOURCE] [--target TARGET]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-7a145563-4146-4768-bf82-fcfafa8a88e1.json


SystemExit: 2

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [6]:
!python build_github_repo.py --source /kaggle/input/github-format-export --target /kaggle/working/cves-stroke-autonomic-complexity

In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Automated GitHub Repository Scaffold Builder
Organizes downloaded Kaggle/Colab outputs and generates all required GitHub
root & documentation files:
    - requirements.txt
    - .gitignore
    - LICENSE (MIT)
    - CITATION.cff
    - docs/DATA_DICTIONARY.md
    - Full folder hierarchy (src/, docs/, results/tables/, results/figures_publication/,
      results/figures_supplementary/, results/supplementary_tachograms/, logs/)

Usage:
    python build_github_repo.py --source ./unzipped_kaggle_outputs --target ./cves-stroke-autonomic-complexity
"""

import argparse
import shutil
import sys
from pathlib import Path

# =============================================================================
# 1. STANDARD GITHUB COMPANION FILE TEMPLATES
# =============================================================================
REQUIREMENTS_TXT = """# Core scientific computing
numpy>=1.24.0
pandas>=2.0.0
scipy>=1.10.0

# Signal processing & PhysioNet WFDB I/O
neurokit2>=0.2.7
wfdb>=4.1.0
awscli>=1.29.0

# Hierarchical mixed-effects modeling & diagnostics
statsmodels>=0.14.0
patsy>=0.5.3

# Publication visualization & progress reporting
matplotlib>=3.7.0
seaborn>=0.13.0
tqdm>=4.65.0
"""

GITIGNORE_TXT = """# Raw PhysioNet CVES binary waveforms & clinical registry (download via AWS S3)
cves-dataset/
*.dat
*.hea
subjects.csv

# Large archives & optional bulk QC plots (~784 PNGs; archive on OSF/Zenodo instead)
*.zip
*.tar.gz
Supplementary_QC_Plots/
results/supplementary_qc_plots/

# Python environments & caches
.venv/
venv/
__pycache__/
*.py[cod]
*$py.class
.pytest_cache/

# Jupyter, Kaggle & IDE metadata
.ipynb_checkpoints/
.virtual_documents/
.vscode/
.idea/

# OS metadata
.DS_Store
Thumbs.db
"""

MIT_LICENSE_TXT = """MIT License

Copyright (c) 2026 Aswin Tony Joseph

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""

CITATION_CFF_TXT = """cff-version: 1.2.0
message: "If you use this pipeline or derived feature tables, please cite this software and the PhysioNet CVES dataset."
title: "Hemispheric Dissociation of Multiscale Autonomic Complexity and Cardiovagal Reserve in Chronic Ischemic Stroke (CCS 1000 Hz Pipeline)"
version: "1.0.0"
date-released: "2026-11-20"
license: MIT
authors:
  - family-names: "Joseph"
    given-names: "Aswin Tony"
    email: "aswintony7@gmail.com"
    affiliation: "Government Medical College Thiruvananthapuram, Kerala, India"
  - family-names: "Venugopal"
    given-names: "Rahul"
    affiliation: "Centre for Consciousness Studies, Department of Neurophysiology, NIMHANS, Bengaluru, India"
keywords:
  - chronic ischemic stroke
  - heart rate variability
  - multiscale entropy
  - deceleration capacity
  - linear mixed-effects models
  - PhysioNet CVES
"""

DATA_DICTIONARY_MD = """# Data Dictionary — Exported Result Tables (`results/tables/`)

## 1. Signal Quality & Epoch Audit Tables
- **`Table_S0_Epoch_Sanity_Audit.csv`**: Automated comparison of extracted `HR_bpm` against `subjects.csv` reference HR columns (`Pearson_r`, `Spearman_rs`, `MAE_bpm`, `Bias_bpm`, `Med_RMSSD_ms`, `Med_DFA_a1`, `Pass_Check`).
- **`Table_S0B_HR_Outliers_gt5bpm.csv`**: Subject-level list of any epoch differing by >5 bpm from `subjects.csv` (includes `Extracted_HR`, `GT_HR`, `Diff_bpm`, `RMSSD_ms`, `SQI`).
- **`Table_S1_Master_Epoch_Features_Nabian2018_1000Hz.csv`**: Full 392-epoch extracted feature table across 75 subjects (114 columns: metadata, SQI, RR quality flags, HR, EDR respiration, PRSA DC/AC, Coarse MSE 6–15, and full NeuroKit2 linear/spectral/nonlinear HRV battery).

## 2. Multicollinearity & LMEM Diagnostic Tables
- **`Table_S2_VIF_Multicollinearity.csv`**: Variance Inflation Factors (`VIF`) and `Tolerance` (`1 / VIF`) across the main-effects LMEM design matrix (all predictors pass `VIF < 5.0`, range `1.003–1.759`).
- **`Table_S3_LMEM_Diagnostics_ICC_R2_Shapiro.csv`**: Random-intercept variance (`Var_u0`), residual variance (`Var_eps`), Intraclass Correlation Coefficient (`ICC`), Nakagawa–Schielzeth Marginal and Conditional $R^2$, REML log-likelihood, and Shapiro–Wilk residual normality statistics across both modeling suites.

## 3. Primary Manuscript Statistical Tables
- **`Table1_Demographics_and_Resting_Subtypes.csv`**: Cohort demographics ($N = 72$) and resting baseline (`Stress_Score == 0`) subtype means and SDs.
- **`Table1B_Resting_Hypothesis_Tests.csv`**: Resting 4-group ANOVAs and Welch/Mann–Whitney pairwise comparisons for H1 (Multiscale Complexity Ratios) and H2 (`PRSA_DC_clean`).
- **`Table2_Task_State_Descriptives.csv`**: Descriptive statistics (`count`, `mean`, `std`, `median`) across all six protocol tasks (States 0–3).
- **`Table3_LMEM_StrictSinus_Orthostatic.csv`**: REML LMEM parameter estimates (`Beta`, `SE`, `z_stat`, `p_value`, 95% CI, `ICC`) across the strict-sinus orthostatic transition ($N = 236$ epochs, $67$ subjects).
- **`Table4_LMEM_4State_Continuum.csv`**: REML LMEM parameter estimates across the calibrated 4-state stress continuum ($N = 377\\text{–}386$ epochs, $72$ subjects).
- **`Table5_Translational_Spearman_Coupling.csv`**: Two-tailed Spearman rank correlations ($r_s$, $R^2$ rank %, $p$-value, $N$) linking resting and Empirical Bayes reactivity slopes (`u1i_*`) to TCD CBFV, MMPF phase shift, Trail Making Part B, 12-min hallway gait velocity, and NIHSS.
"""

# Mapping from Kaggle workspace folder names to clean GitHub repo paths
FOLDER_MAP = {
    "Tables_CSV": "results/tables",
    "Figures_Publication": "results/figures_publication",
    "Figures_Supplementary": "results/figures_supplementary",
    "Supplementary_Tachograms": "results/supplementary_tachograms",
}


# =============================================================================
# 2. REPOSITORY BUILDER LOGIC
# =============================================================================
def create_directory_tree(repo_root: Path) -> None:
    """Creates the standardized GitHub folder hierarchy."""
    subdirs = [
        "src",
        "docs",
        "results/tables",
        "results/figures_publication",
        "results/figures_supplementary",
        "results/supplementary_tachograms",
        "logs",
    ]
    for rel in subdirs:
        (repo_root / rel).mkdir(parents=True, exist_ok=True)


def write_companion_files(repo_root: Path) -> None:
    """Writes requirements.txt, .gitignore, LICENSE, CITATION.cff, and DATA_DICTIONARY.md."""
    files_to_write = {
        "requirements.txt": REQUIREMENTS_TXT,
        ".gitignore": GITIGNORE_TXT,
        "LICENSE": MIT_LICENSE_TXT,
        "CITATION.cff": CITATION_CFF_TXT,
        "docs/DATA_DICTIONARY.md": DATA_DICTIONARY_MD,
    }
    for rel_path, content in files_to_write.items():
        dest = repo_root / rel_path
        dest.write_text(content.strip() + "\n", encoding="utf-8")
        print(f"  [WRITE] {rel_path}")


def copy_kaggle_outputs(source_dir: Path, repo_root: Path) -> None:
    """Copies unzipped Kaggle workspace folders and logs into their clean GitHub paths."""
    if not source_dir.exists():
        print(f"  [WARN] Source directory not found: {source_dir} (created empty scaffold only).")
        return

    # Copy mapped output folders
    for src_sub, dst_rel in FOLDER_MAP.items():
        src_path = source_dir / src_sub
        dst_path = repo_root / dst_rel
        if src_path.exists() and src_path.is_dir():
            count = 0
            for item in sorted(src_path.iterdir()):
                if item.is_file() and not item.name.startswith("."):
                    shutil.copy2(item, dst_path / item.name)
                    count += 1
            print(f"  [COPY]  {src_sub:<26} -> {dst_rel} ({count} files)")

    # Copy run_log.txt if present
    log_src = source_dir / "run_log.txt"
    if log_src.exists():
        shutil.copy2(log_src, repo_root / "logs" / "run_log.txt")
        print("  [COPY]  run_log.txt                -> logs/run_log.txt")

    # Copy pipeline Python scripts if present in source_dir
    for py_file in sorted(source_dir.glob("*.py")):
        if py_file.name != Path(__file__).name:
            shutil.copy2(py_file, repo_root / "src" / py_file.name)
            print(f"  [COPY]  {py_file.name:<26} -> src/{py_file.name}")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Build a clean GitHub repository scaffold for the CVES Stroke HRV project."
    )
    parser.add_argument(
        "--source",
        type=Path,
        default=Path("./cves_nabian2018_ccs_workspace"),
        help="Path to the unzipped Kaggle/Colab workspace folder.",
    )
    parser.add_argument(
        "--target",
        type=Path,
        default=Path("./cves-stroke-autonomic-complexity"),
        help="Target path for the new GitHub repository folder.",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    repo_root = args.target.resolve()
    source_dir = args.source.resolve()

    print("=" * 72)
    print(f"Building GitHub Repository Scaffold at: {repo_root}")
    print("=" * 72)

    try:
        create_directory_tree(repo_root)
        write_companion_files(repo_root)
        copy_kaggle_outputs(source_dir, repo_root)
    except Exception as exc:
        print(f"[ERROR] Failed to construct repository scaffold: {exc}", file=sys.stderr)
        sys.exit(1)

    print("-" * 72)
    print("[DONE] Repository scaffold ready. Next Git commands:")
    print(f"       cd {repo_root}")
    print("       # Place README.md at root, LMEM_Justification_and_Code_Reasoning.md in docs/,")
    print("       # and cves_stroke_hrv_nabian2018_ccs_pipeline_v2.py in src/.")
    print("       git init && git branch -M main")
    print("       git add . && git commit -m 'Initial release: 1000 Hz CCS Nabian2018 stroke HRV pipeline'")
    print("       git remote add origin https://github.com/<your-username>/cves-stroke-autonomic-complexity.git")
    print("       git push -u origin main")
    print("=" * 72)


if __name__ == "__main__":
    main()

usage: colab_kernel_launcher.py [-h] [--source SOURCE] [--target TARGET]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-ccd0a057-40c3-4930-9097-356731e755e9.json


SystemExit: 2